# TER Picopatt - Prediction

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import requests
#from dotenv import load_dotenv

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

import picopatt as fc
from picopatt.io import load_all


In [ ]:
FIG_ROOT = Path("prediction/figures")
FIG_MODELS = FIG_ROOT / "model_analysis"
FIG_ERRORS = FIG_ROOT / "model_analysis"
FIG_DATA   = FIG_ROOT / "prediction_data"

MFDATA = Path("data/processed/prediction/tmrt_pred_report")
MF_RAW = Path("data/processed/meteofrance")

for folder in [FIG_MODELS, FIG_ERRORS, FIG_DATA, MFDATA, MF_RAW]:
    fc.create_folder(folder)

print("FIG_ROOT:", FIG_ROOT)
print("MFDATA:", MFDATA)
print("MF_RAW:", MF_RAW)

In [ ]:
DATA_NOZERO = Path("data/processed/picopatt/clean_nozeros")
bd = load_all(DATA_NOZERO, False)

required_geo = ["lon_ontrack", "lat_ontrack"]
missing_geo = [c for c in required_geo if c not in bd.columns]
if missing_geo:
    raise ValueError("Colonnes GNSS ontrack manquantes: " + ", ".join(missing_geo))

bd = bd.copy()
bd["lon_std"] = pd.to_numeric(bd["lon_ontrack"], errors="coerce")
bd["lat_std"] = pd.to_numeric(bd["lat_ontrack"], errors="coerce")

bd = bd.dropna(subset=["lon_std", "lat_std"])
bd = bd[(bd["lon_std"].between(-180, 180)) & (bd["lat_std"].between(-90, 90))]

bd = bd.reset_index(drop=True)
bd["uid"] = bd.index.astype(int)

print("bd ready:", bd.shape)
print(bd[["uid","lon_std","lat_std"]].head())


In [ ]:
AE = Path("data/processed/alphaearth/alphaearth_data")

# Fichiers exports AlphaEarth
raw_path  = AE / "alphaearth_A00_A63_points.csv"
pca10_path = AE / "alphaearth_pca10_points.csv"
pca15_path = AE / "alphaearth_pca15_points.csv"
pca32_path = AE / "alphaearth_pca32_points.csv"

for p in [raw_path, pca10_path, pca15_path, pca32_path]:
    if not p.exists():
        raise FileNotFoundError(f"Fichier manquant: {p}")

aef_raw  = pd.read_csv(raw_path)
aef_pca10 = pd.read_csv(pca10_path)
aef_pca15 = pd.read_csv(pca15_path)
aef_pca32 = pd.read_csv(pca32_path)

# Noms des colonnes
BANDS = [f"A{i:02d}" for i in range(64)]
PCA10_COLS = [c for c in aef_pca10.columns if c.startswith("aef_pca10_")]
PCA15_COLS = [c for c in aef_pca15.columns if c.startswith("aef_pca15_")]
PCA32_COLS = [c for c in aef_pca32.columns if c.startswith("aef_pca32_")]

# Merge dans bd
df = bd.merge(aef_raw[["uid"] + BANDS], on="uid", how="left")
df = df.merge(aef_pca10[["uid"] + PCA10_COLS], on="uid", how="left")
df = df.merge(aef_pca15[["uid"] + PCA15_COLS], on="uid", how="left")
df = df.merge(aef_pca32[["uid"] + PCA32_COLS], on="uid", how="left")

print("df merged shape:", df.shape)
print("NaN embeddings raw A00:", int(df["A00"].isna().sum()))
print("NaN PCA10:", int(df[PCA10_COLS].isna().any(axis=1).sum()))

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp", "tmrt"]).copy()

start = df["timestamp"].min()
end   = df["timestamp"].max()
print("PICOPATT range:", start, "->", end)

Doit créer un compte sur https://public-api.meteofrance.fr/ puis mettre le token dans .env

In [ ]:
load_dotenv()
TOKEN = os.getenv("MFTOKEN","").strip()
print("Token length:", len(TOKEN))
if len(TOKEN) <= 1400:
    raise ValueError("MFTOKEN trop court. Vérifie .env (et pas tronqué).")

BASE = "https://public-api.meteofrance.fr/public/DPClim/v1"
headers = {"apikey": TOKEN}

params_cmd = {
    "id-station": "34154001",  # MONTPELLIER-AEROPORT
    "date-deb-periode": "2024-10-29T00:00:00Z",
    "date-fin-periode": "2025-01-16T23:59:59Z"
}

r = requests.get(f"{BASE}/commande-station/horaire", headers=headers, params=params_cmd, timeout=120)
print("commande status:", r.status_code, r.text[:200])
r.raise_for_status()

cmd = r.json()
num_cmde = cmd["elaboreProduitAvecDemandeResponse"]["return"]
print("num_cmde:", num_cmde)


In [ ]:
def download_cmd_csv(num_cmde: str, out_path: Path, max_wait_s: int = 900, sleep_s: int = 5):
    t0 = time.time()
    out_path = Path(out_path)

    while True:
        rf = requests.get(
            f"{BASE}/commande/fichier",
            headers=headers,
            params={"id-cmde": num_cmde},
            timeout=120
        )

        if rf.status_code == 201 and len(rf.content) > 0:
            out_path.write_bytes(rf.content)
            return out_path

        if rf.status_code == 204:
            pass
        elif rf.status_code == 410:
            raise RuntimeError("410: production déjà livrée. Relance une commande pour un nouveau fichier.")
        elif rf.status_code == 404:
            raise RuntimeError("404: commande inconnue. Vérifie num_cmde.")
        elif rf.status_code == 507:
            raise RuntimeError("507: production trop volumineuse. Réduis la période.")
        elif rf.status_code >= 400:
            raise RuntimeError(f"Erreur {rf.status_code}: {rf.text[:300]}")

        if time.time() - t0 > max_wait_s:
            raise TimeoutError(f"Fichier non prêt après {max_wait_s}s. Dernier status={rf.status_code}")

        time.sleep(sleep_s)

mf_csv = MF_RAW / f"mf_montpellier_{num_cmde}.csv"
mf_csv = download_cmd_csv(num_cmde, mf_csv)
print("CSV météo écrit:", mf_csv)

In [ ]:
wx = pd.read_csv(MF_RAW / "mf_montpellier_2026004655806.csv", sep=";", dtype=str)

wx["DATE"] = pd.to_datetime(wx["DATE"], format="%Y%m%d%H", utc=True)

num_cols = [c for c in wx.columns if c not in ["POSTE", "DATE"]]
for c in num_cols:
    wx[c] = wx[c].str.replace(",", ".", regex=False).replace({"": None})
    wx[c] = pd.to_numeric(wx[c], errors="coerce")

wx = wx.sort_values("DATE").reset_index(drop=True)

print("wx shape:", wx.shape)
print("wx cols:", len(wx.columns))
display(wx.head(3))

In [ ]:
df2 = df.copy()
df2["timestamp"] = pd.to_datetime(df2["timestamp"], errors="coerce")

# Hypothèse (à garder explicite): timestamp PICOPATT en Europe/Paris si naïf
if df2["timestamp"].dt.tz is None:
    df2["timestamp"] = df2["timestamp"].dt.tz_localize("Europe/Paris")

df2["timestamp_utc"] = df2["timestamp"].dt.tz_convert("UTC")

df2 = df2.sort_values("timestamp_utc")
wx2 = wx.sort_values("DATE")

dfm = pd.merge_asof(
    df2,
    wx2,
    left_on="timestamp_utc",
    right_on="DATE",
    direction="nearest",
    tolerance=pd.Timedelta("30min")
)

print("dfm shape:", dfm.shape)
print("missing MF join:", float(dfm["DATE"].isna().mean()))


In [ ]:
# Variables MF utilisées
# On garde des variables physiques exploitables et on écarte les colonnes Q..., qui sont des codes qualité.
mf_base = [
    "T", "TD", "TN", "TX",          # température, point de rosée, min/max horaires
    "U", "UN", "UX", "UABS",       # humidité relative, min/max et humidité absolue
    "PSTAT", "PMER",                 # pression station / mer
    "FF", "FXY", "FXI",             # vent moyen, max horaire, rafale instantanée
    "RR1", "DRR1",                   # pluie et durée de pluie
    "N", "NBAS",                     # nébulosité totale et couche basse principale
    "INS", "GLO", "GLO2",           # ensoleillement et rayonnement global
    "WW", "VV",                      # temps présent et visibilité
]
mf_base = [c for c in mf_base if c in dfm.columns]

def keep_informative(cols, df=dfm):
    keep, dropped = [], []
    for c in cols:
        values = pd.to_numeric(df[c], errors="coerce")
        if values.notna().any() and values.nunique(dropna=True) > 1:
            keep.append(c)
        else:
            dropped.append(c)
    return keep, dropped

mf_derived_features = []
for c in ["DD", "DXI"]:
    if c in dfm.columns:
        radians = np.deg2rad(pd.to_numeric(dfm[c], errors="coerce") % 360)
        dfm[f"{c}_sin"] = np.sin(radians)
        dfm[f"{c}_cos"] = np.cos(radians)
        mf_derived_features.extend([f"{c}_sin", f"{c}_cos"])

if {"FF", "DD"}.issubset(dfm.columns):
    radians = np.deg2rad(pd.to_numeric(dfm["DD"], errors="coerce") % 360)
    ff = pd.to_numeric(dfm["FF"], errors="coerce")
    dfm["wind_u"] = ff * np.sin(radians)
    dfm["wind_v"] = ff * np.cos(radians)
    mf_derived_features.extend(["wind_u", "wind_v"])

if {"FXI", "DXI"}.issubset(dfm.columns):
    radians = np.deg2rad(pd.to_numeric(dfm["DXI"], errors="coerce") % 360)
    fxi = pd.to_numeric(dfm["FXI"], errors="coerce")
    dfm["gust_u"] = fxi * np.sin(radians)
    dfm["gust_v"] = fxi * np.cos(radians)
    mf_derived_features.extend(["gust_u", "gust_v"])

if {"T", "TD"}.issubset(dfm.columns):
    dfm["dewpoint_depression"] = pd.to_numeric(dfm["T"], errors="coerce") - pd.to_numeric(dfm["TD"], errors="coerce")
    mf_derived_features.append("dewpoint_depression")
if {"TX", "TN"}.issubset(dfm.columns):
    dfm["T_range_hour"] = pd.to_numeric(dfm["TX"], errors="coerce") - pd.to_numeric(dfm["TN"], errors="coerce")
    mf_derived_features.append("T_range_hour")
if {"UX", "UN"}.issubset(dfm.columns):
    dfm["U_range_hour"] = pd.to_numeric(dfm["UX"], errors="coerce") - pd.to_numeric(dfm["UN"], errors="coerce")
    mf_derived_features.append("U_range_hour")
if "RR1" in dfm.columns:
    rr1 = pd.to_numeric(dfm["RR1"], errors="coerce")
    dfm["rain_flag"] = (rr1.fillna(0) > 0).astype(int)
    mf_derived_features.append("rain_flag")
if {"RR1", "DRR1"}.issubset(dfm.columns):
    dfm["rain_duration_weighted"] = pd.to_numeric(dfm["RR1"], errors="coerce").fillna(0) * pd.to_numeric(dfm["DRR1"], errors="coerce").fillna(0)
    mf_derived_features.append("rain_duration_weighted")
if {"GLO", "INS"}.issubset(dfm.columns):
    dfm["GLO_x_INS"] = pd.to_numeric(dfm["GLO"], errors="coerce").fillna(0) * pd.to_numeric(dfm["INS"], errors="coerce").fillna(0)
    mf_derived_features.append("GLO_x_INS")
if "WW" in dfm.columns:
    ww = pd.to_numeric(dfm["WW"], errors="coerce")
    dfm["ww_precip_flag"] = ww.between(50, 99).fillna(False).astype(int)
    dfm["ww_fog_flag"] = ww.between(40, 49).fillna(False).astype(int)
    dfm["ww_thunder_flag"] = ww.between(95, 99).fillna(False).astype(int)
    mf_derived_features.extend(["ww_precip_flag", "ww_fog_flag", "ww_thunder_flag"])

mf_missing_flags = []
for c in mf_base:
    values = pd.to_numeric(dfm[c], errors="coerce")
    if values.notna().any() and values.isna().any():
        flag = f"{c}_missing"
        dfm[flag] = values.isna().astype(int)
        mf_missing_flags.append(flag)

mf_base, dropped_mf_base = keep_informative(mf_base)
mf_derived_features, dropped_mf_derived = keep_informative(mf_derived_features)
mf_missing_flags, dropped_mf_missing = keep_informative(mf_missing_flags)
mf_features = mf_base + mf_derived_features + mf_missing_flags

print("MF base:", mf_base)
print("MF dérivées:", mf_derived_features)
print("MF missing flags:", mf_missing_flags)
print("MF ignorées car absentes/constantes:", dropped_mf_base + dropped_mf_derived + dropped_mf_missing)
print("MF features total:", len(mf_features))

In [ ]:
def drop_all_nan_cols(df_, cols):
    cols = [c for c in cols if c in df_.columns]
    keep = [c for c in cols if df_[c].notna().any()]
    dropped = sorted(set(cols) - set(keep))
    return keep, dropped

TARGET = "tmrt"

ae_pca10 = [c for c in dfm.columns if c.startswith("aef_pca10_")]
ae_raw   = [f"A{i:02d}" for i in range(64) if f"A{i:02d}" in dfm.columns]

train = dfm.dropna(subset=[TARGET, "DATE"]).copy()
train["day_utc"] = train["timestamp_utc"].dt.floor("D")

mf_features, dropped_mf = drop_all_nan_cols(train, mf_features)
ae_pca10, dropped_p10 = drop_all_nan_cols(train, ae_pca10)
ae_raw, dropped_raw = drop_all_nan_cols(train, ae_raw)

print("train:", train.shape)
print("Dropped MF all-NaN:", dropped_mf)
print("Dropped AE PCA10 all-NaN:", dropped_p10)
print("Dropped AE raw all-NaN:", dropped_raw)

In [ ]:
miss = train[mf_base].isna().mean().sort_values(ascending=False)

plt.figure(figsize=(14,4))
plt.bar(miss.index, miss.values)
plt.xticks(rotation=90)
plt.ylabel("Missing rate")
plt.title("MF meteo - missing rate (base variables)")
plt.tight_layout()

out_fig = FIG_DATA / "mf_missing_rate.png"
plt.savefig(out_fig, dpi=150)
plt.show()
print("Figure enregistrée:", out_fig)


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

train["day_utc"] = train["timestamp_utc"].dt.floor("D")
groups = train["day_utc"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

def make_split(X, y):
    tr, te = next(gss.split(X, y, groups=groups))
    return tr, te


In [ ]:
groups = train["day_utc"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

def make_split(X, y):
    tr, te = next(gss.split(X, y, groups=groups))
    return tr, te

In [ ]:
def eval_set(name, feat_cols):
    X = train[feat_cols]
    y = train[TARGET].astype(float)

    tr, te = make_split(X, y)

    model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("reg", HistGradientBoostingRegressor(random_state=42))
    ])

    model.fit(X.iloc[tr], y.iloc[tr])
    pred = model.predict(X.iloc[te])

    return {
        "set": name,
        "rmse": mean_squared_error(y.iloc[te], pred) ** 0.5,
        "mae": mean_absolute_error(y.iloc[te], pred),
        "r2": r2_score(y.iloc[te], pred),
        "n_test": len(te)
    }, (tr, te)

results = []
splits = {}

res, sp = eval_set("MF_only", mf_features)
results.append(res); splits["MF_only"] = sp

if len(ae_pca10) > 0:
    res, sp = eval_set("MF_plus_AE_PCA10", mf_features + ae_pca10)
    results.append(res); splits["MF_plus_AE_PCA10"] = sp

if len(ae_raw) > 0:
    res, sp = eval_set("MF_plus_AE_raw64", mf_features + ae_raw)
    results.append(res); splits["MF_plus_AE_raw64"] = sp

results_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
display(results_df)

out_csv = MFDATA / "tmrt_pred_global_results.csv"
results_df.to_csv(out_csv, index=False)
print("Résultats exportés:", out_csv)

In [ ]:
plt.figure(figsize=(10,4))
plt.bar(results_df["set"], results_df["rmse"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("RMSE (°C)")
plt.title("TMRT prediction - RMSE")
plt.tight_layout()
out_fig = FIG_MODELS / "rmse_by_set.png"
plt.savefig(out_fig, dpi=150)
plt.show()
print("Figure enregistrée:", out_fig)

plt.figure(figsize=(10,4))
plt.bar(results_df["set"], results_df["mae"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("MAE (°C)")
plt.title("TMRT prediction - MAE")
plt.tight_layout()
out_fig = FIG_MODELS / "mae_by_set.png"
plt.savefig(out_fig, dpi=150)
plt.show()
print("Figure enregistrée:", out_fig)

plt.figure(figsize=(10,4))
plt.bar(results_df["set"], results_df["r2"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("R²")
plt.title("TMRT prediction - R²")
plt.tight_layout()
out_fig = FIG_MODELS / "r2_by_set.png"
plt.savefig(out_fig, dpi=150)
plt.show()
print("Figure enregistrée:", out_fig)


In [ ]:
best_set = results_df.iloc[0]["set"]
print("Best set:", best_set)

if best_set == "MF_only":
    feat_cols = mf_features
elif best_set == "MF_plus_AE_PCA10":
    feat_cols = mf_features + ae_pca10
else:
    feat_cols = mf_features + ae_raw

X = train[feat_cols]
y = train[TARGET].astype(float)
tr, te = splits[best_set]

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("reg", HistGradientBoostingRegressor(random_state=42))
])
model.fit(X.iloc[tr], y.iloc[tr])
pred = model.predict(X.iloc[te])

test_rows = train.iloc[te].copy()
test_rows["tmrt_pred"] = pred
test_rows["err"] = test_rows["tmrt_pred"] - test_rows["tmrt"]

print("Test rows:", test_rows.shape)

In [ ]:
pred_csv = MFDATA / f"tmrt_pred_testrows_{best_set}.csv"
test_rows[["uid","timestamp","timestamp_utc","tmrt","tmrt_pred","err","track_id","M_slot","section_id","lon_ontrack","lat_ontrack"]\
         if all(c in test_rows.columns for c in ["uid","track_id","M_slot","section_id","lon_ontrack","lat_ontrack"]) \
         else ["timestamp","timestamp_utc","tmrt","tmrt_pred","err"]].to_csv(pred_csv, index=False)

print("Export test_rows:", pred_csv)

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(test_rows["err"].dropna(), bins=80)
plt.xlabel("err = pred - true (°C)")
plt.ylabel("count")
plt.title(f"Error distribution ({best_set})")
plt.tight_layout()
out_fig = FIG_ERRORS / "error_hist.png"
plt.savefig(out_fig, dpi=150)
plt.show()
print("Figure enregistrée:", out_fig)


In [ ]:
sample = test_rows.sample(n=min(30000, len(test_rows)), random_state=42)

plt.figure(figsize=(5,5))
plt.scatter(sample["tmrt"], sample["tmrt_pred"], s=2)
plt.xlabel("TMRT true")
plt.ylabel("TMRT predicted")
plt.title(f"Predicted vs True ({best_set})")
plt.tight_layout()
out_fig = FIG_ERRORS / "pred_vs_true.png"
plt.savefig(out_fig, dpi=150)
plt.show()
print("Figure enregistrée:", out_fig)


In [ ]:
if {"lon_ontrack","lat_ontrack"}.issubset(test_rows.columns):
    plot_df = test_rows.sample(n=min(20000, len(test_rows)), random_state=42)
    plt.figure(figsize=(7,6))
    plt.scatter(plot_df["lon_ontrack"], plot_df["lat_ontrack"], c=plot_df["err"], s=3)
    plt.colorbar(label="err (°C)")
    plt.xlabel("lon_ontrack")
    plt.ylabel("lat_ontrack")
    plt.title(f"Spatial error map (sample) ({best_set})")
    plt.tight_layout()
    out_fig = FIG_ERRORS / "spatial_error_map.png"
    plt.savefig(out_fig, dpi=150)
    plt.show()
    print("Figure enregistrée:", out_fig)
else:
    print("lon_ontrack/lat_ontrack absents")


In [ ]:
err = test_rows["err"].to_numpy()

summary = {
    "set": best_set,
    "n": int(np.sum(~np.isnan(err))),
    "mean_err": float(np.nanmean(err)),
    "median_err": float(np.nanmedian(err)),
    "mae": float(np.nanmean(np.abs(err))),
    "rmse": float(np.sqrt(np.nanmean(err**2))),
    "p05_abs_err": float(np.nanpercentile(np.abs(err), 5)),
    "p50_abs_err": float(np.nanpercentile(np.abs(err), 50)),
    "p95_abs_err": float(np.nanpercentile(np.abs(err), 95)),
}

summary_df = pd.DataFrame([summary])
display(summary_df)

out_csv = MFDATA / "tmrt_pred_error_summary.csv"
summary_df.to_csv(out_csv, index=False)
print("Résumé exporté:", out_csv)


In [ ]:
group_cols = [c for c in ["track_id", "M_slot", "section_id"] if c in test_rows.columns]
print("group_cols:", group_cols)

def agg_err(g):
    e = g["err"].to_numpy()
    return pd.Series({
        "n": len(e),
        "mae": np.mean(np.abs(e)),
        "rmse": np.sqrt(np.mean(e**2)),
        "bias": np.mean(e),
        "p95_abs": np.percentile(np.abs(e), 95)
    })

if group_cols:
    err_by = (test_rows.groupby(group_cols)
              .apply(agg_err)
              .reset_index()
              .sort_values("rmse", ascending=False))

    display(err_by.head(25))

    out_csv = MFDATA / "tmrt_pred_error_by_groups.csv"
    err_by.to_csv(out_csv, index=False)
    print("Export err_by:", out_csv)

    top = err_by.head(20).copy()
    labels = ["/".join(map(str, row)) for row in top[group_cols].values]

    plt.figure(figsize=(14,4))
    plt.bar(range(len(top)), top["rmse"])
    plt.xticks(range(len(top)), labels, rotation=90)
    plt.ylabel("RMSE (°C)")
    plt.title("Top 20 worst groups by RMSE")
    plt.tight_layout()
    out_fig = FIG_ERRORS / "top20_groups_rmse.png"
    plt.savefig(out_fig, dpi=150)
    plt.show()
    print("Figure enregistrée:", out_fig)

else:
    print("Pas de colonnes track_id / M_slot / section_id")


In [ ]:
def plot_binned_mae(df_, col, q=5):
    tmp = df_.dropna(subset=[col, "err"]).copy()
    if tmp.empty:
        print("Empty:", col)
        return
    tmp["bin"] = pd.qcut(tmp[col], q=q, duplicates="drop")
    tab = tmp.groupby("bin")["err"].apply(lambda x: np.mean(np.abs(x))).reset_index(name="MAE")

    plt.figure(figsize=(10,4))
    plt.bar(tab["bin"].astype(str), tab["MAE"])
    plt.xticks(rotation=25, ha="right")
    plt.ylabel("MAE (°C)")
    plt.title(f"MAE by {col} quantile bins")
    plt.tight_layout()
    out_fig = FIG_ERRORS / f"mae_by_{col}_bins.png"
    plt.savefig(out_fig, dpi=150)
    plt.show()
    print("Figure enregistrée:", out_fig)

for col in ["GLO", "N", "RR1", "FF", "T", "U"]:
    if col in test_rows.columns:
        plot_binned_mae(test_rows, col, q=5)


## Amélioration du modèle TMRT

Axes d'amélioration par rapport au baseline `MF_plus_AE_raw64` :
1. **Features temporelles** — heure + jour de l'année encodés cycliquement (sin/cos)
2. **Position solaire** — élévation, azimut, zénith via `pvlib` (timestamp + GPS)
3. **Features géographiques** — lat/lon du point mesuré
4. **Interactions physiques** — `GLO × elev`, `N × GLO`, `T × GLO`
5. **HistGBR tuné** — 500 itérations, learning_rate=0.05, max_depth=8

In [ ]:
# ── 1. Features temporelles ────────────────────────────────────────────────────
ts = train["timestamp_utc"]

train["hour_sin"] = np.sin(2 * np.pi * ts.dt.hour / 24)
train["hour_cos"] = np.cos(2 * np.pi * ts.dt.hour / 24)
train["doy_sin"]  = np.sin(2 * np.pi * ts.dt.dayofyear / 365)
train["doy_cos"]  = np.cos(2 * np.pi * ts.dt.dayofyear / 365)

time_feats = ["hour_sin", "hour_cos", "doy_sin", "doy_cos"]
print("Features temporelles:", time_feats)

# ── 2. Features géographiques ──────────────────────────────────────────────────
train["lat_feat"] = train["lat_std"]
train["lon_feat"] = train["lon_std"]
geo_feats = ["lat_feat", "lon_feat"]
print("Features géo:", geo_feats)

# ── 3. Position solaire (pvlib) ────────────────────────────────────────────────
solar_feats = []
try:
    import pvlib
    from pvlib.solarposition import get_solarposition
    print("pvlib disponible, calcul position solaire sur", len(train), "points…")

    sol = get_solarposition(
        train["timestamp_utc"],
        train["lat_std"].values,
        train["lon_std"].values,
        method="nrel_numpy"
    )
    train["solar_elevation"] = sol["apparent_elevation"].values
    train["solar_azimuth"]   = sol["azimuth"].values
    train["solar_zenith"]    = sol["apparent_zenith"].values
    solar_feats = ["solar_elevation", "solar_azimuth", "solar_zenith"]
    print("Position solaire OK — élévation médiane:", round(train["solar_elevation"].median(), 2), "°")

except ImportError:
    print("pvlib non installé (pip install pvlib) — proxy angle horaire utilisé")
    train["solar_hour_angle"] = (ts.dt.hour + ts.dt.minute / 60 - 12) * 15
    solar_feats = ["solar_hour_angle"]

# ── 4. Interactions physiques ──────────────────────────────────────────────────
inter_feats = []

if "GLO" in train.columns and "solar_elevation" in train.columns:
    train["GLO_x_elev"] = train["GLO"].fillna(0) * train["solar_elevation"].clip(lower=0)
    inter_feats.append("GLO_x_elev")

if "N" in train.columns and "GLO" in train.columns:
    train["N_x_GLO"] = train["N"].fillna(0) * train["GLO"].fillna(0)
    inter_feats.append("N_x_GLO")

if "T" in train.columns and "GLO" in train.columns:
    train["T_x_GLO"] = train["T"].fillna(0) * train["GLO"].fillna(0)
    inter_feats.append("T_x_GLO")

print("Interactions physiques:", inter_feats)

# ── Récapitulatif ──────────────────────────────────────────────────────────────
extra_feats = time_feats + geo_feats + solar_feats + inter_feats
feat_cols_improved = mf_features + ae_raw + extra_feats

print(f"\nMF features      : {len(mf_features)}")
print(f"AE raw64         : {len(ae_raw)}")
print(f"Extra features   : {len(extra_feats)}  {extra_feats}")
print(f"Total improved   : {len(feat_cols_improved)}")

In [ ]:
# ── Fonction d'évaluation avec HistGBR paramétrable ───────────────────────────
def eval_set_tuned(name, feat_cols, **hgbr_kw):
    X = train[feat_cols]
    y = train[TARGET].astype(float)
    tr, te = make_split(X, y)

    # StandardScaler retiré : inutile pour les arbres gradient
    model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("reg", HistGradientBoostingRegressor(random_state=42, **hgbr_kw))
    ])
    model.fit(X.iloc[tr], y.iloc[tr])
    pred = model.predict(X.iloc[te])

    return {
        "set": name,
        "rmse": mean_squared_error(y.iloc[te], pred) ** 0.5,
        "mae":  mean_absolute_error(y.iloc[te], pred),
        "r2":   r2_score(y.iloc[te], pred),
        "n_test": len(te)
    }, model

hgbr_tuned = dict(
    max_iter=500,
    learning_rate=0.05,
    max_depth=8,
    min_samples_leaf=30,
    l2_regularization=0.1,
)

# ── Comparaison des 4 variantes ────────────────────────────────────────────────
print("=== Évaluation des variantes (patience ~3–5 min) ===\n")

res_all = []

# 1. Baseline : MF + AE64, HistGBR défaut
print("[1/4] Baseline MF+AE64 default…")
r, _ = eval_set_tuned("1_baseline_MF+AE64_default", mf_features + ae_raw)
res_all.append(r); print(f"      RMSE={r['rmse']:.4f}  R²={r['r2']:.4f}")

# 2. MF + AE64 + extra features, HistGBR défaut
print("[2/4] MF+AE64 + extra features, default…")
r, _ = eval_set_tuned("2_MF+AE64+extra_default", feat_cols_improved)
res_all.append(r); print(f"      RMSE={r['rmse']:.4f}  R²={r['r2']:.4f}")

# 3. MF + AE64, HistGBR tuné
print("[3/4] MF+AE64, HistGBR tuné…")
r, _ = eval_set_tuned("3_MF+AE64_tuned", mf_features + ae_raw, **hgbr_tuned)
res_all.append(r); print(f"      RMSE={r['rmse']:.4f}  R²={r['r2']:.4f}")

# 4. MF + AE64 + extra, HistGBR tuné  ← meilleure config attendue
print("[4/4] MF+AE64 + extra + HistGBR tuné…")
r, best_model = eval_set_tuned("4_MF+AE64+extra_tuned", feat_cols_improved, **hgbr_tuned)
res_all.append(r); print(f"      RMSE={r['rmse']:.4f}  R²={r['r2']:.4f}")

# ── Résultats triés ────────────────────────────────────────────────────────────
comp_df = pd.DataFrame(res_all).sort_values("rmse").reset_index(drop=True)
display(comp_df)

baseline_rmse = comp_df[comp_df["set"].str.startswith("1_")]["rmse"].values[0]
best_rmse = comp_df.iloc[0]["rmse"]
print(f"\nBaseline RMSE : {baseline_rmse:.4f} °C")
print(f"Meilleur RMSE : {best_rmse:.4f} °C  ({comp_df.iloc[0]['set']})")
print(f"Gain RMSE     : {baseline_rmse - best_rmse:.4f} °C  ({100*(baseline_rmse - best_rmse)/baseline_rmse:.1f}%)")

out_csv = MFDATA / "tmrt_pred_improved_results.csv"
comp_df.to_csv(out_csv, index=False)
print("\nRésultats exportés:", out_csv)

In [ ]:
# ── Visualisation comparative ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

colors = ["#d9534f" if i > 0 else "#aaaaaa" for i in range(len(comp_df))]
colors_sorted = colors  # already sorted by rmse ascending → last = best

for ax, metric, label in zip(
    axes,
    ["rmse", "mae", "r2"],
    ["RMSE (°C) ↓", "MAE (°C) ↓", "R² ↑"]
):
    vals = comp_df[metric].values[::-1]
    labs = comp_df["set"].values[::-1]
    bar_colors = ["#2196F3" if i == 0 else "#90CAF9" for i in range(len(vals))]
    ax.barh(labs, vals, color=bar_colors)
    ax.set_xlabel(label)
    ax.set_title(label, fontsize=11)
    ax.tick_params(axis="y", labelsize=8)
    for v, lab in zip(vals, range(len(vals))):
        ax.text(v * 0.98 if metric != "r2" else v - 0.005, lab, f"{v:.3f}",
                va="center", ha="right", fontsize=8, color="white", fontweight="bold")

plt.suptitle("Comparaison modèles TMRT — MF + AlphaEarth 64 bandes", fontsize=13)
plt.tight_layout()
out_fig = FIG_MODELS / "comparison_improved.png"
plt.savefig(out_fig, dpi=150)
plt.show()
print("Figure enregistrée:", out_fig)

In [ ]:
# ── Analyse d'erreur sur le meilleur modèle ────────────────────────────────────
X_best = train[feat_cols_improved]
y_best = train[TARGET].astype(float)
tr_best, te_best = make_split(X_best, y_best)

best_model_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("reg", HistGradientBoostingRegressor(random_state=42, **hgbr_tuned))
])
best_model_pipeline.fit(X_best.iloc[tr_best], y_best.iloc[tr_best])
pred_best = best_model_pipeline.predict(X_best.iloc[te_best])

test_best = train.iloc[te_best].copy()
test_best["tmrt_pred"] = pred_best
test_best["err"] = pred_best - test_best[TARGET].values

# Distribution des erreurs
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(test_best["err"].dropna(), bins=80, color="#2196F3", edgecolor="none")
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_xlabel("err = pred − vrai (°C)")
axes[0].set_ylabel("count")
axes[0].set_title(f"Distribution erreurs — best model\nbias={test_best['err'].mean():.2f} °C")

sample = test_best.sample(n=min(30000, len(test_best)), random_state=42)
axes[1].scatter(sample[TARGET], sample["tmrt_pred"], s=2, alpha=0.3, color="#2196F3")
lim = [sample[TARGET].min(), sample[TARGET].max()]
axes[1].plot(lim, lim, "r--", linewidth=1)
axes[1].set_xlabel("TMRT réel (°C)")
axes[1].set_ylabel("TMRT prédit (°C)")
axes[1].set_title("Prédit vs Réel")

plt.tight_layout()
out_fig = FIG_ERRORS / "best_model_error_analysis.png"
plt.savefig(out_fig, dpi=150)
plt.show()
print("Figure enregistrée:", out_fig)

# Résumé chiffré
err = test_best["err"].dropna().to_numpy()
print(f"\nRMSE   : {np.sqrt(np.mean(err**2)):.4f} °C")
print(f"MAE    : {np.mean(np.abs(err)):.4f} °C")
print(f"Bias   : {np.mean(err):.4f} °C")
print(f"P95 |e|: {np.percentile(np.abs(err), 95):.4f} °C")

# Export prédictions
pred_out = MFDATA / "tmrt_pred_testrows_best_improved.csv"
cols_export = ["uid","timestamp","timestamp_utc","tmrt","tmrt_pred","err",
               "lon_ontrack","lat_ontrack","track_id","M_slot","section_id"]
cols_export = [c for c in cols_export if c in test_best.columns]
test_best[cols_export].to_csv(pred_out, index=False)
print("Export prédictions:", pred_out)

In [ ]:
# ── Importance des features (top 30) ──────────────────────────────────────────
hgbr_reg = best_model_pipeline.named_steps["reg"]

if hasattr(hgbr_reg, "feature_importances_"):
    importances = hgbr_reg.feature_importances_
    feat_names = feat_cols_improved

    imp_df = pd.DataFrame({"feature": feat_names, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False).reset_index(drop=True)

    top30 = imp_df.head(30)
    plt.figure(figsize=(10, 8))
    plt.barh(top30["feature"][::-1], top30["importance"][::-1], color="#2196F3")
    plt.xlabel("Importance")
    plt.title("Top 30 features — meilleur modèle (MF+AE64+extra tuné)")
    plt.tight_layout()
    out_fig = FIG_MODELS / "feature_importance_top30.png"
    plt.savefig(out_fig, dpi=150)
    plt.show()
    print("Figure enregistrée:", out_fig)

    print("\nTop 10 features:")
    print(imp_df.head(10).to_string(index=False))
else:
    print("feature_importances_ non disponible pour cet estimateur")

## Toutes les pistes d'amélioration

| Piste | Approche | Objectif |
|---|---|---|
| **P1** | Extra features v2 | Ajouter `is_daytime`, `solar_elev²`, minutes sin/cos, température apparente, interactions |
| **P2** | Hyperparamètres corrigés | Tester 4 configs sans sur-régularisation |
| **P3** | Correction du biais | Shift constant + modèle de résidus (stacking) |
| **P4** | Ensemble | Moyenne de 3 modèles avec seeds différentes |
| **P5** | Permutation importance | Identifier les vraies features importantes |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PISTE 1 — Extra features v2
# ══════════════════════════════════════════════════════════════════════════════

# P1a. Flag jour/nuit
train["is_daytime"] = (train["solar_elevation"] > 0).astype(int)

# P1b. Élévation solaire au carré : effet non-linéaire du rayonnement
train["solar_elev_sq"] = train["solar_elevation"].clip(lower=0) ** 2

# P1c. Précision infra-horaire (minutes) — cycle de 60 min
train["min_sin"] = np.sin(2 * np.pi * train["timestamp_utc"].dt.minute / 60)
train["min_cos"] = np.cos(2 * np.pi * train["timestamp_utc"].dt.minute / 60)

extra_v2 = ["is_daytime", "solar_elev_sq", "min_sin", "min_cos"]

# P1d. Température apparente ressentie (T + humidité)
if {"T", "U"}.issubset(train.columns):
    T_ = train["T"].fillna(train["T"].median())
    U_ = train["U"].fillna(train["U"].median())
    train["apparent_T"] = T_ - 0.4 * (T_ - 10) * (1 - U_ / 100)
    extra_v2.append("apparent_T")

# P1e. T × élévation solaire (chaleur ressentie au soleil)
if "T" in train.columns:
    train["T_x_elev"] = train["T"].fillna(0) * train["solar_elevation"].clip(lower=0)
    extra_v2.append("T_x_elev")

# P1f. Vent × rayonnement (refroidissement éolien sous soleil)
if {"FF", "GLO"}.issubset(train.columns):
    train["FF_x_GLO"] = train["FF"].fillna(0) * train["GLO"].fillna(0)
    extra_v2.append("FF_x_GLO")

# P1g. Nébulosité × élévation (ombre des nuages selon angle)
if "N" in train.columns:
    train["N_x_elev"] = train["N"].fillna(0) * train["solar_elevation"].clip(lower=0)
    extra_v2.append("N_x_elev")

feat_cols_v2 = feat_cols_improved + extra_v2

print(f"Features v2 ajoutées ({len(extra_v2)}) : {extra_v2}")
print(f"Total features v2 : {len(feat_cols_v2)}")

# Évaluation rapide (modèle défaut pour voir l'apport des features seules)
r_p1, _ = eval_set_tuned("P1_v2_features_default", feat_cols_v2)
print(f"\nP1 résultat → RMSE={r_p1['rmse']:.4f}  MAE={r_p1['mae']:.4f}  R²={r_p1['r2']:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PISTE 2 — Hyperparamètres corrigés (sans sur-régularisation)
# ══════════════════════════════════════════════════════════════════════════════
print("=== Piste 2 : Comparaison hyperparamètres sur feat_cols_v2 ===\n")

hgbr_configs = {
    "P2a_no_reg":    dict(max_iter=500, learning_rate=0.05, max_depth=8,  min_samples_leaf=10),
    "P2b_deep":      dict(max_iter=400, learning_rate=0.07, max_depth=10, min_samples_leaf=10),
    "P2c_fast_lr":   dict(max_iter=300, learning_rate=0.10, max_depth=8,  min_samples_leaf=10),
    "P2d_slow_deep": dict(max_iter=600, learning_rate=0.03, max_depth=9,  min_samples_leaf=15),
}

res_p2 = []
best_p2_rmse   = float("inf")
best_p2_params = None
best_p2_name   = None
best_p2_model  = None

for name, params in hgbr_configs.items():
    print(f"  [{name}]…", end=" ", flush=True)
    r, mdl = eval_set_tuned(name, feat_cols_v2, **params)
    res_p2.append(r)
    print(f"RMSE={r['rmse']:.4f}  R²={r['r2']:.4f}")
    if r["rmse"] < best_p2_rmse:
        best_p2_rmse, best_p2_params, best_p2_name, best_p2_model = (
            r["rmse"], params, name, mdl
        )

p2_df = pd.DataFrame(res_p2).sort_values("rmse").reset_index(drop=True)
display(p2_df)
print(f"\nMeilleur config P2 : {best_p2_name} — RMSE={best_p2_rmse:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PISTE 3 — Correction du biais (shift constant + modèle de résidus)
# ══════════════════════════════════════════════════════════════════════════════
print("=== Piste 3 : Correction du biais ===\n")

X_p3 = train[feat_cols_v2]
y_p3 = train[TARGET].astype(float)
tr_p3, te_p3 = make_split(X_p3, y_p3)

# ── 3a. Entraîne le meilleur modèle P2 ────────────────────────────────────────
m_base = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("reg",     HistGradientBoostingRegressor(random_state=42, **best_p2_params))
])
m_base.fit(X_p3.iloc[tr_p3], y_p3.iloc[tr_p3])
pred_base = m_base.predict(X_p3.iloc[te_p3])

rmse_base = mean_squared_error(y_p3.iloc[te_p3], pred_base) ** 0.5
bias_base = float(np.mean(y_p3.iloc[te_p3].values - pred_base))
print(f"Modèle base  → RMSE={rmse_base:.4f}  Bias={bias_base:.4f} °C")

# ── 3b. Correction constante (shift = biais moyen) ────────────────────────────
# Note : en production, estimer le biais sur un jeu de validation séparé
pred_shifted = pred_base + bias_base
rmse_shift = mean_squared_error(y_p3.iloc[te_p3], pred_shifted) ** 0.5
mae_shift  = mean_absolute_error(y_p3.iloc[te_p3], pred_shifted)
r2_shift   = r2_score(y_p3.iloc[te_p3], pred_shifted)
bias_shift = float(np.mean(y_p3.iloc[te_p3].values - pred_shifted))
print(f"Shift constant → RMSE={rmse_shift:.4f}  MAE={mae_shift:.4f}  R²={r2_shift:.4f}  Bias={bias_shift:.4f}")

# ── 3c. Modèle de résidus (stacking propre sans fuite) ────────────────────────
# Divise tr en tr1 (80%) et tr2 (20%)
# → tr1 entraîne le modèle principal
# → tr2 fournit des résidus "non biaisés" pour entraîner le modèle de correction
tr_arr = np.array(tr_p3)
np.random.seed(42)
np.random.shuffle(tr_arr)
n1       = int(0.8 * len(tr_arr))
tr1, tr2 = tr_arr[:n1], tr_arr[n1:]

m1 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("reg",     HistGradientBoostingRegressor(random_state=42, **best_p2_params))
])
m1.fit(X_p3.iloc[tr1], y_p3.iloc[tr1])
resid_tr2 = y_p3.iloc[tr2].values - m1.predict(X_p3.iloc[tr2])
print(f"\nRésidus sur tr2 : mean={resid_tr2.mean():.4f}  std={resid_tr2.std():.4f}")

# Modèle de résidus (plus léger)
m_resid = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("reg",     HistGradientBoostingRegressor(
        random_state=42, max_iter=200, max_depth=5, learning_rate=0.05, min_samples_leaf=20
    ))
])
m_resid.fit(X_p3.iloc[tr2], resid_tr2)

# Rééntraine le modèle principal sur tout tr (pour évaluation finale)
m_final_p3 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("reg",     HistGradientBoostingRegressor(random_state=42, **best_p2_params))
])
m_final_p3.fit(X_p3.iloc[tr_p3], y_p3.iloc[tr_p3])

pred_stk = m_final_p3.predict(X_p3.iloc[te_p3]) + m_resid.predict(X_p3.iloc[te_p3])
rmse_stk = mean_squared_error(y_p3.iloc[te_p3], pred_stk) ** 0.5
mae_stk  = mean_absolute_error(y_p3.iloc[te_p3], pred_stk)
r2_stk   = r2_score(y_p3.iloc[te_p3], pred_stk)
bias_stk = float(np.mean(y_p3.iloc[te_p3].values - pred_stk))
print(f"Modèle résidus → RMSE={rmse_stk:.4f}  MAE={mae_stk:.4f}  R²={r2_stk:.4f}  Bias={bias_stk:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PISTE 4 — Ensemble (moyenne de 3 modèles, seeds différentes)
# ══════════════════════════════════════════════════════════════════════════════
print("=== Piste 4 : Ensemble (3 seeds) ===\n")

X_ens = train[feat_cols_v2]
y_ens = train[TARGET].astype(float)
tr_ens, te_ens = make_split(X_ens, y_ens)

seeds        = [42, 123, 999]
preds_seeds  = []

for seed in seeds:
    print(f"  Entraînement seed={seed}…", end=" ", flush=True)
    mdl = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("reg",     HistGradientBoostingRegressor(random_state=seed, **best_p2_params))
    ])
    mdl.fit(X_ens.iloc[tr_ens], y_ens.iloc[tr_ens])
    p = mdl.predict(X_ens.iloc[te_ens])
    preds_seeds.append(p)
    rmse_s = mean_squared_error(y_ens.iloc[te_ens], p) ** 0.5
    print(f"RMSE={rmse_s:.4f}")

# Moyenne simple des 3 prédictions
pred_ens = np.mean(preds_seeds, axis=0)
rmse_ens = mean_squared_error(y_ens.iloc[te_ens], pred_ens) ** 0.5
mae_ens  = mean_absolute_error(y_ens.iloc[te_ens], pred_ens)
r2_ens   = r2_score(y_ens.iloc[te_ens], pred_ens)
bias_ens = float(np.mean(y_ens.iloc[te_ens].values - pred_ens))

print(f"\nEnsemble (avg)  → RMSE={rmse_ens:.4f}  MAE={mae_ens:.4f}  R²={r2_ens:.4f}  Bias={bias_ens:.4f}")

# Bonus : ensemble + shift constant
pred_ens_shifted = pred_ens + bias_ens
rmse_ens_s = mean_squared_error(y_ens.iloc[te_ens], pred_ens_shifted) ** 0.5
mae_ens_s  = mean_absolute_error(y_ens.iloc[te_ens], pred_ens_shifted)
r2_ens_s   = r2_score(y_ens.iloc[te_ens], pred_ens_shifted)
print(f"Ensemble+shift  → RMSE={rmse_ens_s:.4f}  MAE={mae_ens_s:.4f}  R²={r2_ens_s:.4f}  Bias=0.0000")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PISTE 5 — Permutation importance (remplace feature_importances_ absent)
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.inspection import permutation_importance

print("=== Piste 5 : Permutation importance ===")
print("Calcul sur échantillon de 8 000 points de test (n_repeats=8)…\n")

X_pi = train[feat_cols_v2]
y_pi = train[TARGET].astype(float)
tr_pi, te_pi = make_split(X_pi, y_pi)

# Entraîne le meilleur modèle
m_pi = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("reg",     HistGradientBoostingRegressor(random_state=42, **best_p2_params))
])
m_pi.fit(X_pi.iloc[tr_pi], y_pi.iloc[tr_pi])

# Sous-échantillon du test pour accélérer le calcul
rng    = np.random.default_rng(42)
sample = rng.choice(te_pi, size=min(8000, len(te_pi)), replace=False)

perm = permutation_importance(
    m_pi,
    X_pi.iloc[sample],
    y_pi.iloc[sample],
    n_repeats=8,
    random_state=42,
    n_jobs=-1,
    scoring="neg_root_mean_squared_error"
)

pi_df = pd.DataFrame({
    "feature":    feat_cols_v2,
    "importance": perm.importances_mean,
    "std":        perm.importances_std,
}).sort_values("importance", ascending=False).reset_index(drop=True)

# Top 30
top30 = pi_df.head(30)
fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(top30["feature"][::-1], top30["importance"][::-1],
        xerr=top30["std"][::-1], color="#2196F3", ecolor="gray", capsize=3)
ax.set_xlabel("Importance (augmentation RMSE si permutée)")
ax.set_title("Top 30 features — Permutation Importance")
plt.tight_layout()
out_fig = FIG_MODELS / "permutation_importance_top30.png"
plt.savefig(out_fig, dpi=150)
plt.show()
print("Figure enregistrée:", out_fig)

print("\nTop 15 features :")
print(pi_df.head(15).to_string(index=False))

out_csv = MFDATA / "permutation_importance.csv"
pi_df.to_csv(out_csv, index=False)
print("Export:", out_csv)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# RÉCAPITULATIF FINAL — tous les modèles comparés
# ══════════════════════════════════════════════════════════════════════════════
all_results = [
    # Baseline historique
    {"modele": "0_baseline_MF+AE64",        "rmse": 6.300, "mae": 4.050, "r2": 0.600, "bias": -1.49},
    # Première série d'améliorations (cellules 28)
    {"modele": "1_MF+AE64+extra_default",   "rmse": r_p1["rmse"],  "mae": r_p1["mae"],  "r2": r_p1["r2"],  "bias": None},
    # Piste 2 : meilleur config
    {"modele": f"2_P2_best ({best_p2_name})", "rmse": best_p2_rmse, "mae": None,         "r2": None,        "bias": None},
    # Piste 3a : shift
    {"modele": "3a_P2+shift_constant",       "rmse": rmse_shift,   "mae": mae_shift,    "r2": r2_shift,    "bias": bias_shift},
    # Piste 3b : résidus
    {"modele": "3b_P2+residual_model",       "rmse": rmse_stk,     "mae": mae_stk,      "r2": r2_stk,      "bias": bias_stk},
    # Piste 4 : ensemble
    {"modele": "4_ensemble_3seeds",          "rmse": rmse_ens,     "mae": mae_ens,      "r2": r2_ens,      "bias": bias_ens},
    # Piste 4 bonus : ensemble + shift
    {"modele": "4b_ensemble+shift",          "rmse": rmse_ens_s,   "mae": mae_ens_s,    "r2": r2_ens_s,    "bias": 0.0},
]

# Complète les valeurs manquantes avec un re-calcul si besoin
for d in all_results:
    for k in ["rmse", "mae", "r2", "bias"]:
        if d.get(k) is None:
            d[k] = float("nan")

final_df = (pd.DataFrame(all_results)
              .sort_values("rmse")
              .reset_index(drop=True))

print("=== TOUS LES MODÈLES ===")
display(final_df.round(4))

# Delta vs baseline
base_rmse = 6.300
best_row   = final_df.iloc[0]
print(f"\nBaseline RMSE  : {base_rmse:.4f} °C")
print(f"Meilleur RMSE  : {best_row['rmse']:.4f} °C  ({best_row['modele']})")
print(f"Gain total     : {base_rmse - best_row['rmse']:.4f} °C  ({100*(base_rmse - best_row['rmse'])/base_rmse:.1f}%)")

# ── Graphique final ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plot_df = final_df.dropna(subset=["rmse", "mae", "r2"])

for ax, metric, label in zip(axes, ["rmse", "mae", "r2"], ["RMSE (°C) ↓", "MAE (°C) ↓", "R² ↑"]):
    vals = plot_df[metric].values[::-1]
    labs = plot_df["modele"].values[::-1]
    colors = ["#1565C0" if i == len(vals)-1 else "#42A5F5" for i in range(len(vals))]
    ax.barh(labs, vals, color=colors)
    ax.set_xlabel(label)
    ax.set_title(label, fontsize=11)
    ax.tick_params(axis="y", labelsize=7)
    for v, j in zip(vals, range(len(vals))):
        ax.text(v * 0.99 if metric != "r2" else v - 0.003, j,
                f"{v:.3f}", va="center", ha="right", fontsize=7, color="white", fontweight="bold")

plt.suptitle("Récapitulatif — toutes les pistes d'amélioration TMRT", fontsize=13)
plt.tight_layout()
out_fig = FIG_MODELS / "final_all_models_comparison.png"
plt.savefig(out_fig, dpi=150)
plt.show()
print("Figure enregistrée:", out_fig)

out_csv = MFDATA / "tmrt_all_models_results.csv"
final_df.to_csv(out_csv, index=False)
print("Export CSV:", out_csv)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PISTE 6 — Split spatial fixe par parcours
# train = ecusson | validation = boulevards | test = antigone
# ══════════════════════════════════════════════════════════════════════════════

split_tracks = {
    "train": "ecusson",
    "validation": "boulevards",
    "test": "antigone",
}

available_tracks = set(train["track_id"].dropna().unique())
missing_tracks = sorted(set(split_tracks.values()) - available_tracks)
if missing_tracks:
    raise ValueError(f"track_id manquant(s) pour le split demandé : {missing_tracks}")

feat_cols_grid = feat_cols_v2.copy()

train_mask = train["track_id"].eq(split_tracks["train"])
val_mask   = train["track_id"].eq(split_tracks["validation"])
test_mask  = train["track_id"].eq(split_tracks["test"])

X_train_grid = train.loc[train_mask, feat_cols_grid].copy()
y_train_grid = train.loc[train_mask, TARGET].astype(float).copy()

X_val_grid = train.loc[val_mask, feat_cols_grid].copy()
y_val_grid = train.loc[val_mask, TARGET].astype(float).copy()

X_test_grid = train.loc[test_mask, feat_cols_grid].copy()
y_test_grid = train.loc[test_mask, TARGET].astype(float).copy()

split_summary_df = pd.DataFrame([
    {
        "split": "train",
        "track_id": split_tracks["train"],
        "n_rows": len(X_train_grid),
        "n_days": train.loc[train_mask, "day_utc"].nunique(),
        "tmrt_mean": y_train_grid.mean(),
        "tmrt_std": y_train_grid.std(),
    },
    {
        "split": "validation",
        "track_id": split_tracks["validation"],
        "n_rows": len(X_val_grid),
        "n_days": train.loc[val_mask, "day_utc"].nunique(),
        "tmrt_mean": y_val_grid.mean(),
        "tmrt_std": y_val_grid.std(),
    },
    {
        "split": "test",
        "track_id": split_tracks["test"],
        "n_rows": len(X_test_grid),
        "n_days": train.loc[test_mask, "day_utc"].nunique(),
        "tmrt_mean": y_test_grid.mean(),
        "tmrt_std": y_test_grid.std(),
    },
])

selected_mask = train["track_id"].isin(split_tracks.values())
slot_balance_df = (
    pd.crosstab(
        train.loc[selected_mask, "track_id"],
        train.loc[selected_mask, "M_slot"],
        normalize="index"
    )
    .mul(100)
    .round(1)
)

print("=== Split spatial fixe ===")
display(split_summary_df.round(3))
print("\nRépartition M_slot (% par parcours) :")
display(slot_balance_df)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PISTE 7 — Grid search sur train=ecusson, validation=boulevards
# ══════════════════════════════════════════════════════════════════════════════

from sklearn.model_selection import GridSearchCV, PredefinedSplit

X_search = pd.concat([X_train_grid, X_val_grid], axis=0).reset_index(drop=True)
y_search = pd.concat([y_train_grid, y_val_grid], axis=0).reset_index(drop=True)

predefined_fold = np.r_[
    np.full(len(X_train_grid), -1, dtype=int),   # train
    np.zeros(len(X_val_grid), dtype=int),        # validation
]
spatial_cv = PredefinedSplit(test_fold=predefined_fold)

fixed_hgbr_params = {
    "l2_regularization": 0.1,
}

param_grid = {
    "reg__max_iter": [300, 500],
    "reg__learning_rate": [0.03, 0.05],
    "reg__max_depth": [6, 8],
    "reg__min_samples_leaf": [10, 20],
}

grid_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("reg", HistGradientBoostingRegressor(random_state=42, **fixed_hgbr_params))
])

n_combinations = int(np.prod([len(v) for v in param_grid.values()]))

print("=== Grid search spatial ===")
print(f"Train       : {split_tracks['train']} ({len(X_train_grid):,} lignes)")
print(f"Validation  : {split_tracks['validation']} ({len(X_val_grid):,} lignes)")
print(f"Features    : {len(feat_cols_grid)}")
print(f"Combinaisons: {n_combinations}")

grid = GridSearchCV(
    estimator=grid_model,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=spatial_cv,
    refit=False,
    return_train_score=True,
    n_jobs=1,
    verbose=2,
)

grid.fit(X_search, y_search)

grid_results_df = (
    pd.DataFrame(grid.cv_results_)
    .assign(
        rmse_val=lambda d: -d["mean_test_score"],
        rmse_train=lambda d: -d["mean_train_score"],
    )
    .sort_values("rmse_val")
    .reset_index(drop=True)
)

best_grid_params = {
    **fixed_hgbr_params,
    **{k.replace("reg__", ""): v for k, v in grid.best_params_.items()}
}
best_grid_rmse_val = float(grid_results_df.loc[0, "rmse_val"])

display(
    grid_results_df[
        [
            "rank_test_score",
            "rmse_val",
            "rmse_train",
            "param_reg__max_iter",
            "param_reg__learning_rate",
            "param_reg__max_depth",
            "param_reg__min_samples_leaf",
        ]
    ].head(10).round(4)
)

print("\nMeilleurs hyperparamètres :", best_grid_params)
print(f"RMSE validation (boulevards) : {best_grid_rmse_val:.4f} °C")

out_csv = MFDATA / "tmrt_grid_search_spatial_results.csv"
grid_results_df.to_csv(out_csv, index=False)
print("Export grid search :", out_csv)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PISTE 8 — Entraînement final sur ecusson, évaluation sur boulevards + antigone
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_split(split_name, X, y, model):
    pred = model.predict(X)
    return {
        "split": split_name,
        "track_id": split_tracks[split_name],
        "rmse": mean_squared_error(y, pred) ** 0.5,
        "mae": mean_absolute_error(y, pred),
        "r2": r2_score(y, pred),
        "bias": float(np.mean(y.to_numpy() - pred)),
        "n": len(y),
    }, pred

spatial_best_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("reg", HistGradientBoostingRegressor(random_state=42, **best_grid_params))
])

spatial_best_model.fit(X_train_grid, y_train_grid)

val_metrics, val_pred = evaluate_split("validation", X_val_grid, y_val_grid, spatial_best_model)
test_metrics, test_pred = evaluate_split("test", X_test_grid, y_test_grid, spatial_best_model)

spatial_results_df = pd.DataFrame([val_metrics, test_metrics])
display(spatial_results_df.round(4))

print(f"Validation ({split_tracks['validation']}) RMSE : {val_metrics['rmse']:.4f} °C")
print(f"Test final ({split_tracks['test']}) RMSE       : {test_metrics['rmse']:.4f} °C")

export_cols = [
    c for c in [
        "uid", "timestamp", "timestamp_utc", TARGET,
        "track_id", "M_slot", "section_id", "lon_ontrack", "lat_ontrack"
    ]
    if c in train.columns
]

val_pred_df = train.loc[val_mask, export_cols].copy()
val_pred_df["tmrt_pred"] = val_pred
val_pred_df["err"] = val_pred - y_val_grid.to_numpy()

test_pred_df = train.loc[test_mask, export_cols].copy()
test_pred_df["tmrt_pred"] = test_pred
test_pred_df["err"] = test_pred - y_test_grid.to_numpy()

val_out = MFDATA / "tmrt_pred_boulevards_validation_spatial.csv"
test_out = MFDATA / "tmrt_pred_antigone_test_spatial.csv"
metrics_out = MFDATA / "tmrt_spatial_split_metrics.csv"

val_pred_df.to_csv(val_out, index=False)
test_pred_df.to_csv(test_out, index=False)
spatial_results_df.to_csv(metrics_out, index=False)

print("\nExports :")
print(" -", val_out)
print(" -", test_out)
print(" -", metrics_out)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PISTE 6 — Split spatio-temporel par jours au sein de chaque quartier
# bloc = (track_id, day_utc)
# ══════════════════════════════════════════════════════════════════════════════

from sklearn.model_selection import train_test_split

feat_cols_grid = feat_cols_v2.copy()

train = train.copy()
train["st_day_block"] = (
    train["track_id"].astype(str)
    + "__" + train["day_utc"].dt.strftime("%Y-%m-%d")
)

day_blocks = (
    train[["track_id", "day_utc", "st_day_block"]]
    .drop_duplicates()
    .sort_values(["track_id", "day_utc"])
    .reset_index(drop=True)
)

split_map = {}

for track_name, g in day_blocks.groupby("track_id", sort=False):
    blocks = g["st_day_block"].to_numpy()
    
    if len(blocks) < 5:
        raise ValueError(
            f"{track_name} n'a que {len(blocks)} jours, insuffisant pour un split train/val/test."
        )

    # 20% test
    blocks_trainval, blocks_test = train_test_split(
        blocks,
        test_size=0.20,
        random_state=42,
        shuffle=True,
    )

    # 25% de trainval = 20% total pour validation
    blocks_train, blocks_val = train_test_split(
        blocks_trainval,
        test_size=0.25,
        random_state=42,
        shuffle=True,
    )

    for b in blocks_train:
        split_map[b] = "train"
    for b in blocks_val:
        split_map[b] = "validation"
    for b in blocks_test:
        split_map[b] = "test"

train["st_split"] = train["st_day_block"].map(split_map)

if train["st_split"].isna().any():
    raise ValueError("Certains blocs jour/quartier n'ont pas reçu de split.")

train_mask = train["st_split"].eq("train")
val_mask   = train["st_split"].eq("validation")
test_mask  = train["st_split"].eq("test")

X_train_grid = train.loc[train_mask, feat_cols_grid].copy()
y_train_grid = train.loc[train_mask, TARGET].astype(float).copy()

X_val_grid = train.loc[val_mask, feat_cols_grid].copy()
y_val_grid = train.loc[val_mask, TARGET].astype(float).copy()

X_test_grid = train.loc[test_mask, feat_cols_grid].copy()
y_test_grid = train.loc[test_mask, TARGET].astype(float).copy()

split_summary_df = (
    train.groupby("st_split")
    .agg(
        n_rows=(TARGET, "size"),
        n_day_blocks=("st_day_block", "nunique"),
        n_days=("day_utc", "nunique"),
        n_tracks=("track_id", "nunique"),
        tmrt_mean=(TARGET, "mean"),
        tmrt_std=(TARGET, "std"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)

track_balance_df = (
    pd.crosstab(train["st_split"], train["track_id"], normalize="index")
    .mul(100)
    .round(1)
)

slot_balance_df = (
    pd.crosstab(train["st_split"], train["M_slot"], normalize="index")
    .mul(100)
    .round(1)
)

block_split_df = (
    day_blocks.assign(split=day_blocks["st_day_block"].map(split_map))
    .sort_values(["split", "track_id", "day_utc"])
    .reset_index(drop=True)
)

block_balance_df = (
    pd.crosstab(block_split_df["split"], block_split_df["track_id"])
    .reindex(index=["train", "validation", "test"])
)

print("=== Split spatio-temporel (bloc = quartier + jour) ===")
display(split_summary_df.round(3))

print("\nRépartition des blocs jour/quartier :")
display(block_balance_df)

print("\nRépartition des lignes par quartier (%) :")
display(track_balance_df)

print("\nRépartition des lignes par M_slot (%) :")
display(slot_balance_df)

print("\nDétail des blocs affectés :")
display(block_split_df)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PISTE 7 — Grid search avec validation spatio-temporelle fixe
# ══════════════════════════════════════════════════════════════════════════════

from sklearn.model_selection import GridSearchCV, PredefinedSplit

X_search = pd.concat([X_train_grid, X_val_grid], axis=0).reset_index(drop=True)
y_search = pd.concat([y_train_grid, y_val_grid], axis=0).reset_index(drop=True)

predefined_fold = np.r_[
    np.full(len(X_train_grid), -1, dtype=int),   # train
    np.zeros(len(X_val_grid), dtype=int),        # validation
]
st_cv = PredefinedSplit(test_fold=predefined_fold)

fixed_hgbr_params = {
    "l2_regularization": 0.1,
}

param_grid = {
    "reg__max_iter": [300, 500],
    "reg__learning_rate": [0.03, 0.05],
    "reg__max_depth": [6, 8],
    "reg__min_samples_leaf": [10, 20],
}

grid_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("reg", HistGradientBoostingRegressor(random_state=42, **fixed_hgbr_params))
])

n_combinations = int(np.prod([len(v) for v in param_grid.values()]))

print("=== Grid search spatio-temporel ===")
print(f"Train       : {len(X_train_grid):,} lignes")
print(f"Validation  : {len(X_val_grid):,} lignes")
print(f"Test        : {len(X_test_grid):,} lignes")
print(f"Features    : {len(feat_cols_grid)}")
print(f"Combinaisons: {n_combinations}")

grid = GridSearchCV(
    estimator=grid_model,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=st_cv,
    refit=False,
    return_train_score=True,
    n_jobs=1,
    verbose=2,
)

grid.fit(X_search, y_search)

grid_results_df = (
    pd.DataFrame(grid.cv_results_)
    .assign(
        rmse_val=lambda d: -d["mean_test_score"],
        rmse_train=lambda d: -d["mean_train_score"],
    )
    .sort_values("rmse_val")
    .reset_index(drop=True)
)

best_grid_params = {
    **fixed_hgbr_params,
    **{k.replace("reg__", ""): v for k, v in grid.best_params_.items()}
}
best_grid_rmse_val = float(grid_results_df.loc[0, "rmse_val"])

display(
    grid_results_df[
        [
            "rank_test_score",
            "rmse_val",
            "rmse_train",
            "param_reg__max_iter",
            "param_reg__learning_rate",
            "param_reg__max_depth",
            "param_reg__min_samples_leaf",
        ]
    ].head(10).round(4)
)

print("\nMeilleurs hyperparamètres :", best_grid_params)
print(f"RMSE validation spatio-temporelle : {best_grid_rmse_val:.4f} °C")

out_csv = MFDATA / "tmrt_grid_search_spatiotemporal_results.csv"
grid_results_df.to_csv(out_csv, index=False)
print("Export grid search :", out_csv)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PISTE 8 — Entraînement final + évaluation sur le test spatio-temporel
# ══════════════════════════════════════════════════════════════════════════════

def compute_metrics(y_true, pred):
    y_true = np.asarray(y_true, dtype=float)
    pred = np.asarray(pred, dtype=float)
    return {
        "rmse": mean_squared_error(y_true, pred) ** 0.5,
        "mae": mean_absolute_error(y_true, pred),
        "r2": r2_score(y_true, pred),
        "bias": float(np.mean(y_true - pred)),
    }

spatiotemp_best_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("reg", HistGradientBoostingRegressor(random_state=42, **best_grid_params))
])

spatiotemp_best_model.fit(X_train_grid, y_train_grid)

val_pred = spatiotemp_best_model.predict(X_val_grid)
test_pred = spatiotemp_best_model.predict(X_test_grid)

val_metrics = {
    "split": "validation",
    **compute_metrics(y_val_grid, val_pred),
    "n": len(y_val_grid),
}
test_metrics = {
    "split": "test",
    **compute_metrics(y_test_grid, test_pred),
    "n": len(y_test_grid),
}

spatiotemp_results_df = pd.DataFrame([val_metrics, test_metrics])
display(spatiotemp_results_df.round(4))

print(f"Validation RMSE : {val_metrics['rmse']:.4f} °C")
print(f"Test final RMSE : {test_metrics['rmse']:.4f} °C")

export_cols = [
    c for c in [
        "uid", "timestamp", "timestamp_utc", TARGET,
        "track_id", "M_slot", "day_utc", "section_id",
        "lon_ontrack", "lat_ontrack", "st_block", "st_split"
    ]
    if c in train.columns
]

val_pred_df = train.loc[val_mask, export_cols].copy()
val_pred_df["tmrt_pred"] = val_pred
val_pred_df["err"] = val_pred - y_val_grid.to_numpy()

test_pred_df = train.loc[test_mask, export_cols].copy()
test_pred_df["tmrt_pred"] = test_pred
test_pred_df["err"] = test_pred - y_test_grid.to_numpy()

# Détail test par quartier
per_track_test_df = pd.DataFrame([
    {
        "track_id": track_name,
        **compute_metrics(g[TARGET], g["tmrt_pred"]),
        "n": len(g),
    }
    for track_name, g in test_pred_df.groupby("track_id")
]).sort_values("rmse")

print("\nDétail test par quartier :")
display(per_track_test_df.round(4))

val_out = MFDATA / "tmrt_pred_validation_spatiotemporal.csv"
test_out = MFDATA / "tmrt_pred_test_spatiotemporal.csv"
metrics_out = MFDATA / "tmrt_spatiotemporal_split_metrics.csv"
per_track_out = MFDATA / "tmrt_spatiotemporal_test_by_track.csv"

val_pred_df.to_csv(val_out, index=False)
test_pred_df.to_csv(test_out, index=False)
spatiotemp_results_df.to_csv(metrics_out, index=False)
per_track_test_df.to_csv(per_track_out, index=False)

print("\nExports :")
print(" -", val_out)
print(" -", test_out)
print(" -", metrics_out)
print(" -", per_track_out)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PISTE 9 — Robustesse du split spatio-temporel (splits répétés)
# On garde best_grid_params fixe, et on fait varier seulement le tirage des blocs
# bloc = (track_id, day_utc)
# ══════════════════════════════════════════════════════════════════════════════

from sklearn.model_selection import train_test_split

try:
    params_for_repeats = best_grid_params.copy()
except NameError:
    params_for_repeats = {
        "l2_regularization": 0.1,
        "learning_rate": 0.05,
        "max_depth": 8,
        "max_iter": 500,
        "min_samples_leaf": 20,
    }

feat_cols_repeat = feat_cols_v2.copy()
seeds = list(range(10))   # change en range(5) si tu veux aller plus vite

def compute_metrics(y_true, pred):
    y_true = np.asarray(y_true, dtype=float)
    pred = np.asarray(pred, dtype=float)
    return {
        "rmse": mean_squared_error(y_true, pred) ** 0.5,
        "mae": mean_absolute_error(y_true, pred),
        "r2": r2_score(y_true, pred),
        "bias": float(np.mean(y_true - pred)),
    }

def make_spatiotemporal_day_split(df, random_state):
    df = df.copy()

    df["st_day_block"] = (
        df["track_id"].astype(str)
        + "__" + df["day_utc"].dt.strftime("%Y-%m-%d")
    )

    day_blocks = (
        df[["track_id", "day_utc", "st_day_block"]]
        .drop_duplicates()
        .sort_values(["track_id", "day_utc"])
        .reset_index(drop=True)
    )

    split_map = {}

    for track_name, g in day_blocks.groupby("track_id", sort=False):
        blocks = g["st_day_block"].to_numpy()

        if len(blocks) < 5:
            raise ValueError(
                f"{track_name} n'a que {len(blocks)} jours, insuffisant pour un split train/val/test."
            )

        blocks_trainval, blocks_test = train_test_split(
            blocks,
            test_size=0.20,
            random_state=random_state,
            shuffle=True,
        )

        blocks_train, blocks_val = train_test_split(
            blocks_trainval,
            test_size=0.25,   # 0.25 de 80% = 20% total
            random_state=random_state,
            shuffle=True,
        )

        for b in blocks_train:
            split_map[b] = "train"
        for b in blocks_val:
            split_map[b] = "validation"
        for b in blocks_test:
            split_map[b] = "test"

    df["st_split"] = df["st_day_block"].map(split_map)

    if df["st_split"].isna().any():
        raise ValueError("Certains blocs jour/quartier n'ont pas reçu de split.")

    block_assign_df = day_blocks.copy()
    block_assign_df["st_split"] = block_assign_df["st_day_block"].map(split_map)

    return df, block_assign_df

repeat_results = []
repeat_test_track_results = []
repeat_block_results = []

for rs in seeds:
    split_df, block_assign_df = make_spatiotemporal_day_split(train, rs)

    train_mask = split_df["st_split"].eq("train")
    val_mask   = split_df["st_split"].eq("validation")
    test_mask  = split_df["st_split"].eq("test")

    X_train_rep = split_df.loc[train_mask, feat_cols_repeat]
    y_train_rep = split_df.loc[train_mask, TARGET].astype(float)

    X_val_rep = split_df.loc[val_mask, feat_cols_repeat]
    y_val_rep = split_df.loc[val_mask, TARGET].astype(float)

    X_test_rep = split_df.loc[test_mask, feat_cols_repeat]
    y_test_rep = split_df.loc[test_mask, TARGET].astype(float)

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("reg", HistGradientBoostingRegressor(random_state=42, **params_for_repeats))
    ])
    model.fit(X_train_rep, y_train_rep)

    for split_name, X_part, y_part in [
        ("validation", X_val_rep, y_val_rep),
        ("test", X_test_rep, y_test_rep),
    ]:
        pred = model.predict(X_part)
        metrics = compute_metrics(y_part, pred)

        repeat_results.append({
            "random_state": rs,
            "split": split_name,
            "rmse": metrics["rmse"],
            "mae": metrics["mae"],
            "r2": metrics["r2"],
            "bias": metrics["bias"],
            "n": len(y_part),
        })

        if split_name == "test":
            tmp = split_df.loc[test_mask, ["track_id", TARGET]].copy()
            tmp["tmrt_pred"] = pred

            for track_name, g in tmp.groupby("track_id"):
                track_metrics = compute_metrics(g[TARGET], g["tmrt_pred"])
                repeat_test_track_results.append({
                    "random_state": rs,
                    "track_id": track_name,
                    "rmse": track_metrics["rmse"],
                    "mae": track_metrics["mae"],
                    "r2": track_metrics["r2"],
                    "bias": track_metrics["bias"],
                    "n": len(g),
                })

    block_count_df = (
        block_assign_df.groupby(["st_split", "track_id"])
        .size()
        .reset_index(name="n_day_blocks")
    )
    block_count_df["random_state"] = rs
    repeat_block_results.append(block_count_df)

repeat_results_df = pd.DataFrame(repeat_results)
repeat_test_track_df = pd.DataFrame(repeat_test_track_results)
repeat_block_df = pd.concat(repeat_block_results, ignore_index=True)

repeat_summary_df = (
    repeat_results_df.groupby("split")[["rmse", "mae", "r2", "bias"]]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)

repeat_test_track_summary_df = (
    repeat_test_track_df.groupby("track_id")[["rmse", "mae", "r2", "bias"]]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)

print("=== Résultats par split et par seed ===")
display(repeat_results_df.round(4))

print("\n=== Synthèse globale ===")
display(repeat_summary_df)

print("\n=== Test par quartier et par seed ===")
display(repeat_test_track_df.round(4))

print("\n=== Synthèse test par quartier ===")
display(repeat_test_track_summary_df)

print("\n=== Répartition des blocs jour/quartier par seed ===")
display(
    repeat_block_df
    .sort_values(["random_state", "st_split", "track_id"])
    .reset_index(drop=True)
)

# Petit graphe RMSE validation/test selon le seed
plot_df = repeat_results_df.pivot(index="random_state", columns="split", values="rmse")
ax = plot_df.plot(marker="o", figsize=(8, 4))
ax.set_title("RMSE selon le random_state")
ax.set_ylabel("RMSE (°C)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Exports
repeat_results_out = MFDATA / "tmrt_spatiotemporal_repeated_split_results.csv"
repeat_test_track_out = MFDATA / "tmrt_spatiotemporal_repeated_test_by_track.csv"
repeat_block_out = MFDATA / "tmrt_spatiotemporal_repeated_block_counts.csv"

repeat_summary_out = MFDATA / "tmrt_spatiotemporal_repeated_summary.csv"
repeat_test_track_summary_out = MFDATA / "tmrt_spatiotemporal_repeated_test_by_track_summary.csv"

repeat_results_df.to_csv(repeat_results_out, index=False)
repeat_test_track_df.to_csv(repeat_test_track_out, index=False)
repeat_block_df.to_csv(repeat_block_out, index=False)

repeat_summary_export = repeat_summary_df.copy()
repeat_summary_export.columns = [f"{a}_{b}" for a, b in repeat_summary_export.columns]
repeat_summary_export = repeat_summary_export.reset_index()
repeat_summary_export.to_csv(repeat_summary_out, index=False)

repeat_test_track_summary_export = repeat_test_track_summary_df.copy()
repeat_test_track_summary_export.columns = [f"{a}_{b}" for a, b in repeat_test_track_summary_export.columns]
repeat_test_track_summary_export = repeat_test_track_summary_export.reset_index()
repeat_test_track_summary_export.to_csv(repeat_test_track_summary_out, index=False)

print("\nExports :")
print(" -", repeat_results_out)
print(" -", repeat_test_track_out)
print(" -", repeat_block_out)
print(" -", repeat_summary_out)
print(" -", repeat_test_track_summary_out)


In [ ]:
%pip install catboost


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PISTE 10 — CatBoost sur le split spatio-temporel courant
# ══════════════════════════════════════════════════════════════════════════════

try:
    from catboost import CatBoostRegressor, Pool
except ImportError as e:
    raise ImportError("CatBoost n'est pas installé. Lance `%pip install catboost` puis relance.") from e

from sklearn.model_selection import ParameterGrid

def score_reg(y_true, pred):
    y_true = np.asarray(y_true, dtype=float)
    pred = np.asarray(pred, dtype=float)
    return {
        "rmse": mean_squared_error(y_true, pred) ** 0.5,
        "mae": mean_absolute_error(y_true, pred),
        "r2": r2_score(y_true, pred),
        "bias": float(np.mean(y_true - pred)),
    }

# Features numériques déjà construites
feat_cols_num_cb = feat_cols_v2.copy()

# Catégorielles spatiales / contextuelles
cat_cols_cb = [c for c in ["track_id", "M_slot", "section_id", "segment_id"] if c in train.columns]

# Variante de départ : numériques + catégorielles
feat_cols_cb = feat_cols_num_cb + [c for c in cat_cols_cb if c not in feat_cols_num_cb]

X_train_cb = train.loc[train_mask, feat_cols_cb].copy()
y_train_cb = train.loc[train_mask, TARGET].astype(float).copy()

X_val_cb = train.loc[val_mask, feat_cols_cb].copy()
y_val_cb = train.loc[val_mask, TARGET].astype(float).copy()

X_test_cb = train.loc[test_mask, feat_cols_cb].copy()
y_test_cb = train.loc[test_mask, TARGET].astype(float).copy()

# CatBoost attend des catégorielles en str
for X_part in [X_train_cb, X_val_cb, X_test_cb]:
    for c in cat_cols_cb:
        X_part[c] = X_part[c].fillna("__NA__").astype(str)

pool_train_cb = Pool(X_train_cb, y_train_cb, cat_features=cat_cols_cb)
pool_val_cb   = Pool(X_val_cb, y_val_cb, cat_features=cat_cols_cb)
pool_test_cb  = Pool(X_test_cb, y_test_cb, cat_features=cat_cols_cb)

print("=== CatBoost setup ===")
print(f"Train      : {len(X_train_cb):,} lignes")
print(f"Validation : {len(X_val_cb):,} lignes")
print(f"Test       : {len(X_test_cb):,} lignes")
print(f"Num feats  : {len(feat_cols_num_cb)}")
print(f"Cat feats  : {cat_cols_cb}")
print(f"Total feats: {len(feat_cols_cb)}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Grid search léger CatBoost sur validation spatio-temporelle
# ══════════════════════════════════════════════════════════════════════════════

param_grid_cb = {
    "depth": [6, 8],
    "learning_rate": [0.03, 0.05],
    "l2_leaf_reg": [3, 8],
}

cb_results = []
best_cb_model = None
best_cb_params = None
best_cb_val_rmse = float("inf")

for params in ParameterGrid(param_grid_cb):
    print("Test config:", params)

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        iterations=2000,
        random_seed=42,
        allow_writing_files=False,
        thread_count=-1,
        **params,
    )

    model.fit(
        pool_train_cb,
        eval_set=pool_val_cb,
        use_best_model=True,
        early_stopping_rounds=100,
        verbose=False,
    )

    val_pred = model.predict(X_val_cb)
    val_metrics = score_reg(y_val_cb, val_pred)

    cb_results.append({
        **params,
        "best_iteration": int(model.get_best_iteration()),
        "rmse_val": val_metrics["rmse"],
        "mae_val": val_metrics["mae"],
        "r2_val": val_metrics["r2"],
        "bias_val": val_metrics["bias"],
    })

    if val_metrics["rmse"] < best_cb_val_rmse:
        best_cb_val_rmse = val_metrics["rmse"]
        best_cb_params = {
            **params,
            "best_iteration": int(model.get_best_iteration()),
        }
        best_cb_model = model

cb_results_df = pd.DataFrame(cb_results).sort_values("rmse_val").reset_index(drop=True)
display(cb_results_df.round(4))

print("\nMeilleure config CatBoost :", best_cb_params)
print(f"RMSE validation CatBoost : {best_cb_val_rmse:.4f} °C")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Évaluation finale CatBoost sur validation + test
# ══════════════════════════════════════════════════════════════════════════════

val_pred_cb = best_cb_model.predict(X_val_cb)
test_pred_cb = best_cb_model.predict(X_test_cb)

val_metrics_cb = {"split": "validation", **score_reg(y_val_cb, val_pred_cb), "n": len(y_val_cb)}
test_metrics_cb = {"split": "test", **score_reg(y_test_cb, test_pred_cb), "n": len(y_test_cb)}

cb_eval_df = pd.DataFrame([val_metrics_cb, test_metrics_cb])
display(cb_eval_df.round(4))

print(f"Validation RMSE CatBoost : {val_metrics_cb['rmse']:.4f} °C")
print(f"Test final RMSE CatBoost : {test_metrics_cb['rmse']:.4f} °C")

# Détail test par quartier
test_pred_cb_df = train.loc[test_mask, ["track_id", TARGET]].copy()
test_pred_cb_df["tmrt_pred"] = test_pred_cb

cb_test_by_track_df = pd.DataFrame([
    {
        "track_id": track_name,
        **score_reg(g[TARGET], g["tmrt_pred"]),
        "n": len(g),
    }
    for track_name, g in test_pred_cb_df.groupby("track_id")
]).sort_values("rmse").reset_index(drop=True)

print("\nDétail test CatBoost par quartier :")
display(cb_test_by_track_df.round(4))

# Feature importance
cb_fi_df = (
    pd.DataFrame({
        "feature": feat_cols_cb,
        "importance": best_cb_model.get_feature_importance(pool_train_cb),
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("\nTop 25 features CatBoost :")
display(cb_fi_df.head(25).round(4))

# Exports
export_cols_cb = [
    c for c in [
        "uid", "timestamp", "timestamp_utc", TARGET,
        "track_id", "M_slot", "day_utc", "section_id",
        "segment_id", "lon_ontrack", "lat_ontrack",
        "st_day_block", "st_split"
    ]
    if c in train.columns
]

val_export_cb = train.loc[val_mask, export_cols_cb].copy()
val_export_cb["tmrt_pred"] = val_pred_cb
val_export_cb["err"] = val_pred_cb - y_val_cb.to_numpy()

test_export_cb = train.loc[test_mask, export_cols_cb].copy()
test_export_cb["tmrt_pred"] = test_pred_cb
test_export_cb["err"] = test_pred_cb - y_test_cb.to_numpy()

cb_results_path = MFDATA / "tmrt_catboost_grid_results.csv"
cb_eval_path = MFDATA / "tmrt_catboost_eval.csv"
cb_track_path = MFDATA / "tmrt_catboost_test_by_track.csv"
cb_val_pred_path = MFDATA / "tmrt_catboost_pred_validation.csv"
cb_test_pred_path = MFDATA / "tmrt_catboost_pred_test.csv"
cb_fi_path = MFDATA / "tmrt_catboost_feature_importance.csv"

cb_results_df.to_csv(cb_results_path, index=False)
cb_eval_df.to_csv(cb_eval_path, index=False)
cb_test_by_track_df.to_csv(cb_track_path, index=False)
val_export_cb.to_csv(cb_val_pred_path, index=False)
test_export_cb.to_csv(cb_test_pred_path, index=False)
cb_fi_df.to_csv(cb_fi_path, index=False)

print("\nExports :")
print(" -", cb_results_path)
print(" -", cb_eval_path)
print(" -", cb_track_path)
print(" -", cb_val_pred_path)
print(" -", cb_test_pred_path)
print(" -", cb_fi_path)


## Piste 11 ? R?duire l'autocorr?lation et tester `WW` en cat?gorielle

Cette exp?rience v?rifie deux points importants pour l'?valuation du mod?le :

- le split al?atoire est optimiste quand des points tr?s proches dans l'espace et le temps se retrouvent ? la fois en apprentissage et en test ;
- agr?ger les points par tron?ons de 10 m ou 20 m par passage r?duit le poids artificiel des paquets de mesures quasi identiques.

On compare donc les points bruts, une agr?gation 10 m et une agr?gation 20 m, avec trois strat?gies d'?valuation : split al?atoire, split par passage, et split par bloc `track_id + jour`. On teste aussi `WW` comme variable cat?gorielle CatBoost au lieu de la traiter comme une grandeur continue.



In [ ]:
# ??????????????????????????????????????????????????????????????????????????????
# PISTE 11 ? Non-IID, agr?gation spatiale 10/20 m et WW cat?goriel
# ??????????????????????????????????????????????????????????????????????????????

from pathlib import Path
import sys

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import prediction.heatmap_montpellier_10m as hm

OUT_DIR = ROOT / "data" / "processed" / "prediction" / "tmrt_pred_report"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET = hm.TARGET

CATBOOST_PARAMS_NONIID = dict(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=700,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=8,
    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    thread_count=-1,
    verbose=False,
)


def haversine_m(lon1, lat1, lon2, lat2):
    lon1 = np.deg2rad(lon1)
    lat1 = np.deg2rad(lat1)
    lon2 = np.deg2rad(lon2)
    lat2 = np.deg2rad(lat2)
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6_371_000 * 2 * np.arcsin(np.sqrt(a))


def passage_columns(df):
    cols = [c for c in ["track_id", "day_utc", "M_slot", "fichier_originaire"] if c in df.columns]
    if not cols:
        raise ValueError("Aucune colonne de passage disponible.")
    return cols


def passage_id(df):
    cols = passage_columns(df)
    return df[cols].astype(str).agg("__".join, axis=1)


def day_track_id(df):
    cols = [c for c in ["track_id", "day_utc"] if c in df.columns]
    if len(cols) < 2:
        return passage_id(df)
    return df[cols].astype(str).agg("__".join, axis=1)


def add_distance_bins(df, bin_m):
    sort_cols = passage_columns(df) + ["timestamp_utc"]
    work = df.sort_values([c for c in sort_cols if c in df.columns]).reset_index(drop=True).copy()
    lon = hm.safe_numeric(work["lon_std"]).to_numpy(dtype=float)
    lat = hm.safe_numeric(work["lat_std"]).to_numpy(dtype=float)
    step = np.zeros(len(work), dtype=float)
    cumdist = np.zeros(len(work), dtype=float)

    for _, positions in work.groupby(passage_columns(work), sort=False).indices.items():
        pos = np.asarray(positions, dtype=int)
        if len(pos) <= 1:
            continue
        d = haversine_m(lon[pos[:-1]], lat[pos[:-1]], lon[pos[1:]], lat[pos[1:]])
        d = np.where(np.isfinite(d), d, 0.0)
        step[pos[1:]] = d
        cumdist[pos] = np.r_[0.0, np.cumsum(d)]

    work["distance_step_m"] = step
    work["distance_m_passage"] = cumdist
    work["distance_bin_m"] = np.floor(cumdist / float(bin_m)).astype(int)
    return work


def first_non_null(series):
    non_null = series.dropna()
    return non_null.iloc[0] if len(non_null) else np.nan


def mode_or_first(series):
    non_null = series.dropna()
    if not len(non_null):
        return np.nan
    mode = non_null.mode(dropna=True)
    return mode.iloc[0] if len(mode) else non_null.iloc[0]


def aggregate_by_distance(df, bin_m):
    if bin_m is None:
        out = df.copy().reset_index(drop=True)
        out["n_points_aggregated"] = 1
        out["aggregation_m"] = 0
        return out

    binned = add_distance_bins(df, bin_m)
    group_cols = passage_columns(binned) + ["distance_bin_m"]
    code_like = {"WW", "W1", "W2", "DD", "DXI", "DXY", "section_id", "segment_id"}
    exclude_numeric = set(group_cols) | code_like | {"uid"}

    agg_map = {}
    numeric_cols = binned.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if col not in exclude_numeric:
            agg_map[col] = "mean"

    for col in code_like:
        if col in binned.columns and col not in group_cols:
            agg_map[col] = mode_or_first

    for col in ["timestamp", "timestamp_utc", "DATE", "date"]:
        if col in binned.columns and col not in group_cols:
            agg_map[col] = first_non_null

    grouped = binned.groupby(group_cols, dropna=False, sort=False)
    out = grouped.agg(agg_map).reset_index()
    out["n_points_aggregated"] = grouped.size().to_numpy()
    out["aggregation_m"] = int(bin_m)
    return out.reset_index(drop=True)


def prepare_features(df, ww_as_category):
    work = df.copy()
    work, mf_derived = hm.add_meteofrance_derived_features(work)
    mf_features = hm.build_mf_features(work, mf_derived)

    cat_features = []
    if ww_as_category and "WW" in work.columns:
        mf_features = [c for c in mf_features if c != "WW"]
        ww = hm.safe_numeric(work["WW"]).round().astype("Int64")
        work["WW_cat"] = ww.astype(str).replace("<NA>", "__NA__")
        cat_features = ["WW_cat"]

    work, extra_features = hm.add_time_and_extra_features(work, include_coordinates=False)
    raw_features = [c for c in hm.BANDS if c in work.columns and hm.safe_numeric(work[c]).notna().any()]
    numeric_features = mf_features + raw_features + extra_features
    numeric_features = hm.keep_informative_numeric(work, [c for c in numeric_features if c in work.columns])
    feature_cols = numeric_features + cat_features

    X = work[feature_cols].copy()
    for col in numeric_features:
        X[col] = hm.safe_numeric(X[col])
    for col in cat_features:
        X[col] = X[col].fillna("__NA__").astype(str)

    meta = {
        "n_mf_features": len(mf_features) + len(cat_features),
        "n_alphaearth_raw_features": len(raw_features),
        "n_extra_features": len(extra_features),
        "n_total_features": len(feature_cols),
        "mf_features": mf_features + cat_features,
        "cat_features": cat_features,
    }
    return work, X, feature_cols, cat_features, meta


def assign_split(df, strategy):
    split = pd.Series(index=df.index, dtype="object")

    if strategy == "random_row":
        idx = np.arange(len(df))
        train_val, test = train_test_split(idx, test_size=0.20, random_state=RANDOM_STATE, shuffle=True)
        train, validation = train_test_split(train_val, test_size=0.25, random_state=RANDOM_STATE, shuffle=True)
        split.iloc[train] = "train"
        split.iloc[validation] = "validation"
        split.iloc[test] = "test"
        return split

    if strategy == "passage_group":
        groups = passage_id(df)
    elif strategy == "day_track_group":
        groups = day_track_id(df)
    else:
        raise ValueError(strategy)

    unique_groups = pd.Series(groups.dropna().unique())
    train_val_groups, test_groups = train_test_split(
        unique_groups, test_size=0.20, random_state=RANDOM_STATE, shuffle=True
    )
    train_groups, validation_groups = train_test_split(
        train_val_groups, test_size=0.25, random_state=RANDOM_STATE, shuffle=True
    )

    split.loc[groups.isin(train_groups)] = "train"
    split.loc[groups.isin(validation_groups)] = "validation"
    split.loc[groups.isin(test_groups)] = "test"
    if split.isna().any():
        raise ValueError(f"Split incomplet pour {strategy}")
    return split


def score_reg(y_true, pred):
    y_true = np.asarray(y_true, dtype=float)
    pred = np.asarray(pred, dtype=float)
    return {
        "rmse": mean_squared_error(y_true, pred) ** 0.5,
        "mae": mean_absolute_error(y_true, pred),
        "r2": r2_score(y_true, pred) if len(np.unique(y_true)) > 1 else np.nan,
        "bias_pred_minus_obs": float(np.mean(pred - y_true)),
    }


def evaluate_predictions(df_part, y_true, pred):
    point_scores = score_reg(y_true, pred)
    tmp = pd.DataFrame({
        "passage_id": passage_id(df_part).to_numpy(),
        "tmrt_obs": np.asarray(y_true, dtype=float),
        "tmrt_pred": np.asarray(pred, dtype=float),
    })
    by_passage = tmp.groupby("passage_id", as_index=False)[["tmrt_obs", "tmrt_pred"]].mean()
    passage_scores = score_reg(by_passage["tmrt_obs"], by_passage["tmrt_pred"])
    return {
        "point_rmse": point_scores["rmse"],
        "point_mae": point_scores["mae"],
        "point_r2": point_scores["r2"],
        "point_bias_pred_minus_obs": point_scores["bias_pred_minus_obs"],
        "passage_rmse": passage_scores["rmse"],
        "passage_mae": passage_scores["mae"],
        "passage_r2": passage_scores["r2"],
        "passage_bias_pred_minus_obs": passage_scores["bias_pred_minus_obs"],
        "n_passages_eval": int(by_passage["passage_id"].nunique()),
    }


def fit_and_evaluate(dataset_name, df, ww_as_category, split_strategy):
    work, X, feature_cols, cat_features, meta = prepare_features(df, ww_as_category=ww_as_category)
    split = assign_split(work, split_strategy)
    y = hm.safe_numeric(work[TARGET]).astype(float)

    masks = {name: split.eq(name) for name in ["train", "validation", "test"]}
    pool_train = Pool(X.loc[masks["train"]], y.loc[masks["train"]], cat_features=cat_features)
    pool_val = Pool(X.loc[masks["validation"]], y.loc[masks["validation"]], cat_features=cat_features)

    model = CatBoostRegressor(**CATBOOST_PARAMS_NONIID)
    model.fit(
        pool_train,
        eval_set=pool_val,
        use_best_model=True,
        early_stopping_rounds=80,
        verbose=False,
    )

    rows = []
    for split_name in ["validation", "test"]:
        mask = masks[split_name]
        pool_eval = Pool(X.loc[mask], cat_features=cat_features)
        pred = model.predict(pool_eval)
        scores = evaluate_predictions(work.loc[mask], y.loc[mask], pred)
        rows.append({
            "dataset": dataset_name,
            "ww_mode": "categorical" if ww_as_category else "numeric",
            "split_strategy": split_strategy,
            "split": split_name,
            "n_train_rows": int(masks["train"].sum()),
            "n_validation_rows": int(masks["validation"].sum()),
            "n_eval_rows": int(mask.sum()),
            "n_total_rows_dataset": int(len(work)),
            "mean_points_per_agg_row": float(work.get("n_points_aggregated", pd.Series([1])).mean()),
            "best_iteration": int(model.get_best_iteration() or CATBOOST_PARAMS_NONIID["iterations"]),
            **meta,
            **scores,
        })
    return rows


print("Chargement des donn?es PICOPATT + AlphaEarth + M?t?o-France...")
base = hm.load_picopatt_with_alphaearth(ROOT)
wx = hm.load_meteofrance(ROOT)
dfm = hm.merge_nearest_weather(base, wx)
train_base = dfm.dropna(subset=[TARGET, "DATE"]).copy().reset_index(drop=True)
train_base["day_utc"] = train_base["timestamp_utc"].dt.floor("D")
print(f"Donn?es point par point : {len(train_base):,} lignes")

DATASETS = {
    "points_1s": aggregate_by_distance(train_base, None),
    "agg_10m": aggregate_by_distance(train_base, 10),
    "agg_20m": aggregate_by_distance(train_base, 20),
}
for name, df_dataset in DATASETS.items():
    print(
        f"{name}: {len(df_dataset):,} lignes "
        f"(moyenne {df_dataset['n_points_aggregated'].mean():.2f} points agr?g?s/ligne)"
    )

EXPERIMENTS = [
    ("points_1s", False),
    ("points_1s", True),
    ("agg_10m", True),
    ("agg_20m", True),
]
SPLIT_STRATEGIES = ["random_row", "passage_group", "day_track_group"]

all_rows = []
for dataset_name, ww_as_category in EXPERIMENTS:
    for split_strategy in SPLIT_STRATEGIES:
        print(f"\n[RUN] dataset={dataset_name} | WW={'cat' if ww_as_category else 'num'} | split={split_strategy}")
        rows = fit_and_evaluate(dataset_name, DATASETS[dataset_name], ww_as_category, split_strategy)
        all_rows.extend(rows)
        display(pd.DataFrame(rows).round(4))

non_iid_eval_df = pd.DataFrame(all_rows)
main_cols = [
    "dataset", "ww_mode", "split_strategy", "split",
    "n_total_rows_dataset", "n_eval_rows", "n_passages_eval",
    "mean_points_per_agg_row", "n_total_features", "best_iteration",
    "point_rmse", "point_mae", "point_r2", "point_bias_pred_minus_obs",
    "passage_rmse", "passage_mae", "passage_r2", "passage_bias_pred_minus_obs",
]
non_iid_eval_df = non_iid_eval_df[main_cols + [c for c in non_iid_eval_df.columns if c not in main_cols]]

out_csv = OUT_DIR / "tmrt_catboost_non_iid_aggregation_ww_eval.csv"
non_iid_eval_df.to_csv(out_csv, index=False)
print("\nExport :", out_csv)
display(non_iid_eval_df.round(4))

# Synth?se lisible : test seulement, tri?e par strat?gie d'?valuation.
test_summary = (
    non_iid_eval_df[non_iid_eval_df["split"].eq("test")]
    .sort_values(["split_strategy", "point_rmse"])
    .reset_index(drop=True)
)
print("\nSynth?se test :")
display(test_summary[[
    "dataset", "ww_mode", "split_strategy",
    "n_total_rows_dataset", "n_eval_rows", "n_passages_eval",
    "point_rmse", "point_mae", "point_r2",
    "passage_rmse", "passage_mae", "passage_r2",
]].round(4))



### R?sultat de la piste 11

Les r?sultats sont export?s dans `data/processed/prediction/tmrt_pred_report/tmrt_catboost_non_iid_aggregation_ww_eval.csv`.

Points importants :

- le split al?atoire reste tr?s optimiste : sur les points bruts, le RMSE test est proche de 3,13 ?C ;
- le split par passage est plus r?aliste : le RMSE test passe autour de 5,10 ?C sur les points bruts ;
- le split par bloc `track_id + jour` est le plus s?v?re : le RMSE test est autour de 7,47 ?C sur les points bruts ;
- l'agr?gation spatiale r?duit fortement la redondance : 341 281 points bruts deviennent 22 185 lignes ? 10 m et 11 224 lignes ? 20 m ;
- sur le RMSE point avec split group?, l'agr?gation ? 20 m aide l?g?rement (`passage_group` : 5,10 -> 4,85 ; `day_track_group` : 7,47 -> 6,77), mais les m?triques moyenn?es par passage ne s'am?liorent pas syst?matiquement ;
- `WW` en cat?gorielle am?liore tr?s l?g?rement le split al?atoire, mais n'am?liore pas les splits robustes. Pour le mod?le final, `WW` num?rique + flags m?t?o reste donc plus prudent.

Interpr?tation : le mod?le ne ? triche ? pas volontairement, mais le split al?atoire m?lange des points presque identiques entre apprentissage et test. Il mesure donc en partie la capacit? ? interpoler localement dans un m?me passage, pas la g?n?ralisation ? de nouveaux jours/passages.



## Piste 12 ? Analyse d?taill?e des erreurs du CatBoost

Cette section regarde o? le mod?le se trompe sur une ?valuation robuste. On r?entra?ne le CatBoost avec les variables m?t?o enrichies et les embeddings AlphaEarth, puis on ?value uniquement un test par blocs `track_id + jour`.

Les erreurs sont analys?es par parcours, par cr?neau `M_slot`, par niveau de TMRT observ?e, par m?t?o, et par profils d'environnement AlphaEarth. Les profils AlphaEarth ne sont pas une v?rit? terrain ? urbain/ouvert ? : ce sont des clusters d'embeddings, ordonn?s selon la TMRT observ?e sous conditions ensoleill?es pour donner un proxy ombrag?/dense vs ouvert/expos?.



In [ ]:
# ??????????????????????????????????????????????????????????????????????????????
# PISTE 12 ? Analyse d?taill?e des erreurs du CatBoost
# ??????????????????????????????????????????????????????????????????????????????

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import prediction.heatmap_montpellier_10m as hm

OUT_DIR = ROOT / "data" / "processed" / "prediction" / "error_analysis"
FIG_DIR = ROOT / "prediction" / "figures" / "model_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET = hm.TARGET
TRACK_ORDER = ["antigone", "boulevards", "ecusson"]
MSLOT_ORDER = ["M1", "M2", "M3", "M4"]

CATBOOST_PARAMS_ERROR_ANALYSIS = dict(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=700,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=8,
    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    thread_count=-1,
    verbose=False,
)


def day_track_split(df):
    blocks = (
        df[["track_id", "day_utc"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["track_id", "day_utc"])
        .reset_index(drop=True)
    )
    blocks["block"] = blocks["track_id"].astype(str) + "__" + blocks["day_utc"].dt.strftime("%Y-%m-%d")
    split_map = {}
    for _, g in blocks.groupby("track_id", sort=False):
        values = g["block"].to_numpy()
        train_val, test = train_test_split(values, test_size=0.20, random_state=RANDOM_STATE, shuffle=True)
        train, validation = train_test_split(train_val, test_size=0.25, random_state=RANDOM_STATE, shuffle=True)
        split_map.update({b: "train" for b in train})
        split_map.update({b: "validation" for b in validation})
        split_map.update({b: "test" for b in test})
    row_blocks = df["track_id"].astype(str) + "__" + df["day_utc"].dt.strftime("%Y-%m-%d")
    split = row_blocks.map(split_map)
    if split.isna().any():
        raise ValueError("Split day_track incomplet.")
    return split


def prepare_model_matrix(df):
    work = df.copy()
    work, mf_derived = hm.add_meteofrance_derived_features(work)
    mf_features = hm.build_mf_features(work, mf_derived)
    work, extra_features = hm.add_time_and_extra_features(work, include_coordinates=False)
    raw_features = [c for c in hm.BANDS if c in work.columns and hm.safe_numeric(work[c]).notna().any()]
    feature_cols = hm.keep_informative_numeric(work, mf_features + raw_features + extra_features)
    X = work[feature_cols].apply(hm.safe_numeric).replace([np.inf, -np.inf], np.nan)
    meta = {
        "n_mf_features": len(mf_features),
        "n_alphaearth_raw_features": len(raw_features),
        "n_extra_features": len(extra_features),
        "n_total_features": len(feature_cols),
    }
    return work, X, feature_cols, meta


def score_reg(y_true, pred):
    y_true = np.asarray(y_true, dtype=float)
    pred = np.asarray(pred, dtype=float)
    out = {
        "n": int(len(y_true)),
        "tmrt_obs_mean": float(np.mean(y_true)),
        "tmrt_pred_mean": float(np.mean(pred)),
        "bias_pred_minus_obs": float(np.mean(pred - y_true)),
        "mae": float(mean_absolute_error(y_true, pred)),
        "rmse": float(mean_squared_error(y_true, pred) ** 0.5),
        "p90_abs_error": float(np.percentile(np.abs(pred - y_true), 90)),
    }
    out["r2"] = float(r2_score(y_true, pred)) if len(np.unique(y_true)) > 1 else np.nan
    return out


def group_metrics(df, group_cols):
    rows = []
    for keys, g in df.groupby(group_cols, dropna=False, sort=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        row.update(score_reg(g[TARGET], g["tmrt_pred"]))
        rows.append(row)
    return pd.DataFrame(rows).sort_values("rmse", ascending=False).reset_index(drop=True)


def save_table(df, name):
    path = OUT_DIR / name
    df.to_csv(path, index=False)
    print("Export table:", path)
    return path


def save_fig(fig, name):
    path = FIG_DIR / name
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Export figure:", path)
    return path


def add_error_categories(test_df):
    out = test_df.copy()
    out["abs_err"] = out["err"].abs()

    out["tmrt_level"] = pd.qcut(
        out[TARGET],
        q=5,
        duplicates="drop",
    ).astype(str)

    glo = hm.safe_numeric(out.get("GLO", pd.Series(index=out.index)))
    out["glo_level"] = pd.cut(
        glo,
        bins=[-np.inf, 0, 25, 75, 125, np.inf],
        labels=["0_nul", "1_faible", "2_moyen", "3_fort", "4_tres_fort"],
    ).astype(str)

    t = hm.safe_numeric(out.get("T", pd.Series(index=out.index)))
    out["air_temp_level"] = pd.qcut(t, q=4, duplicates="drop").astype(str)

    rr1 = hm.safe_numeric(out.get("RR1", pd.Series(index=out.index))).fillna(0)
    out["rain_state"] = np.where(rr1 > 0, "pluie", "sans_pluie")

    ff = hm.safe_numeric(out.get("FF", pd.Series(index=out.index)))
    out["wind_level"] = pd.cut(
        ff,
        bins=[-np.inf, 2, 5, 8, np.inf],
        labels=["0_calme", "1_modere", "2_venteux", "3_tres_venteux"],
    ).astype(str)

    n = hm.safe_numeric(out.get("N", pd.Series(index=out.index)))
    out["cloud_level"] = "N_manquant"
    out.loc[n <= 2, "cloud_level"] = "peu_nuageux"
    out.loc[(n > 2) & (n <= 6), "cloud_level"] = "nuageux"
    out.loc[(n > 6) & (n <= 8), "cloud_level"] = "tres_nuageux"
    out.loc[n >= 9, "cloud_level"] = "ciel_invisible_ou_brouillard"

    ww = hm.safe_numeric(out.get("WW", pd.Series(index=out.index)))
    out["ww_class"] = "autre"
    out.loc[ww.eq(0), "ww_class"] = "temps_present_nul"
    out.loc[ww.between(40, 49), "ww_class"] = "brouillard"
    out.loc[ww.between(50, 69), "ww_class"] = "pluie_ou_bruine"
    out.loc[ww.between(70, 79), "ww_class"] = "neige"
    out.loc[ww.between(95, 99), "ww_class"] = "orage"

    out["weather_context"] = out["glo_level"].astype(str) + "__" + out["rain_state"].astype(str) + "__" + out["ww_class"].astype(str)
    return out


def add_alphaearth_environment_proxy(test_df):
    out = test_df.copy()
    bands = [c for c in hm.BANDS if c in out.columns]
    X_env = out[bands].apply(hm.safe_numeric)
    X_env = X_env.fillna(X_env.median(numeric_only=True))

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=min(8, len(bands)), random_state=RANDOM_STATE)),
        ("kmeans", KMeans(n_clusters=5, n_init=20, random_state=RANDOM_STATE)),
    ])
    clusters = pipe.fit_predict(X_env)
    out["alphaearth_cluster"] = clusters

    sunny = hm.safe_numeric(out.get("GLO", pd.Series(index=out.index))).fillna(0) >= 50
    ref = out.loc[sunny].copy()
    if ref.empty:
        ref = out.copy()

    cluster_order = (
        ref.groupby("alphaearth_cluster")[TARGET]
        .median()
        .sort_values()
        .index.tolist()
    )
    labels = [
        "1_proxy_ombrage_dense",
        "2_proxy_plutot_ombrage",
        "3_proxy_intermediaire",
        "4_proxy_plutot_ouvert",
        "5_proxy_ouvert_expose",
    ]
    label_map = {cluster: labels[i] for i, cluster in enumerate(cluster_order)}
    out["environment_proxy"] = out["alphaearth_cluster"].map(label_map)

    desc = group_metrics(out, ["environment_proxy"])
    cluster_profile = (
        out.groupby("environment_proxy")
        .agg(
            alphaearth_cluster=("alphaearth_cluster", "first"),
            lon_median=("lon_std", "median"),
            lat_median=("lat_std", "median"),
            sunny_share=("GLO", lambda s: float((hm.safe_numeric(s).fillna(0) >= 50).mean())),
            tmrt_sunny_median=(TARGET, lambda s: float(s.loc[sunny.reindex(s.index).fillna(False)].median()) if sunny.reindex(s.index).fillna(False).any() else np.nan),
            dominant_track=("track_id", lambda s: s.value_counts().index[0]),
            dominant_mslot=("M_slot", lambda s: s.value_counts().index[0]),
        )
        .reset_index()
    )
    desc = desc.merge(cluster_profile, on="environment_proxy", how="left")
    return out, desc


print("Chargement des donn?es...")
base = hm.load_picopatt_with_alphaearth(ROOT)
wx = hm.load_meteofrance(ROOT)
dfm = hm.merge_nearest_weather(base, wx)
train_base = dfm.dropna(subset=[TARGET, "DATE"]).copy().reset_index(drop=True)
train_base["day_utc"] = train_base["timestamp_utc"].dt.floor("D")
print(f"Donn?es disponibles : {len(train_base):,} lignes")

work, X, feature_cols, feature_meta = prepare_model_matrix(train_base)
y = hm.safe_numeric(work[TARGET]).astype(float)
split = day_track_split(work)

train_mask = split.eq("train")
val_mask = split.eq("validation")
test_mask = split.eq("test")
print("Split:", split.value_counts().to_dict())
print("Features:", feature_meta)

model = CatBoostRegressor(**CATBOOST_PARAMS_ERROR_ANALYSIS)
model.fit(
    X.loc[train_mask],
    y.loc[train_mask],
    eval_set=(X.loc[val_mask], y.loc[val_mask]),
    use_best_model=True,
    early_stopping_rounds=80,
    verbose=False,
)

pred_test = model.predict(X.loc[test_mask])
test_errors = work.loc[test_mask].copy()
test_errors["tmrt_pred"] = pred_test
test_errors["err"] = test_errors["tmrt_pred"] - hm.safe_numeric(test_errors[TARGET])
test_errors = add_error_categories(test_errors)
test_errors, env_desc = add_alphaearth_environment_proxy(test_errors)

# Exports ligne par ligne et tables d'analyse.
export_cols = [
    c for c in [
        "uid", "timestamp", "timestamp_utc", "track_id", "M_slot", "day_utc", "section_id", "segment_id",
        "lon_ontrack", "lat_ontrack", "lon_std", "lat_std", TARGET, "tmrt_pred", "err", "abs_err",
        "T", "GLO", "INS", "RR1", "FF", "N", "WW", "tmrt_level", "glo_level", "air_temp_level",
        "rain_state", "wind_level", "cloud_level", "ww_class", "weather_context",
        "alphaearth_cluster", "environment_proxy",
    ]
    if c in test_errors.columns
]
save_table(test_errors[export_cols], "catboost_day_track_test_predictions_with_errors.csv")

overall = pd.DataFrame([{**score_reg(test_errors[TARGET], test_errors["tmrt_pred"]), **feature_meta, "best_iteration": int(model.get_best_iteration() or 0)}])
by_track = group_metrics(test_errors, ["track_id"])
by_mslot = group_metrics(test_errors, ["M_slot"])
by_track_mslot = group_metrics(test_errors, ["track_id", "M_slot"])
by_tmrt = group_metrics(test_errors, ["tmrt_level"])
by_glo = group_metrics(test_errors, ["glo_level"])
by_temp = group_metrics(test_errors, ["air_temp_level"])
by_rain = group_metrics(test_errors, ["rain_state"])
by_wind = group_metrics(test_errors, ["wind_level"])
by_cloud = group_metrics(test_errors, ["cloud_level"])
by_ww = group_metrics(test_errors, ["ww_class"])
by_weather_context = group_metrics(test_errors, ["weather_context"])
by_env = group_metrics(test_errors, ["environment_proxy"])

save_table(overall, "catboost_error_overall.csv")
save_table(by_track, "catboost_error_by_track.csv")
save_table(by_mslot, "catboost_error_by_mslot.csv")
save_table(by_track_mslot, "catboost_error_by_track_mslot.csv")
save_table(by_tmrt, "catboost_error_by_tmrt_level.csv")
save_table(by_glo, "catboost_error_by_glo_level.csv")
save_table(by_temp, "catboost_error_by_air_temp_level.csv")
save_table(by_rain, "catboost_error_by_rain_state.csv")
save_table(by_wind, "catboost_error_by_wind_level.csv")
save_table(by_cloud, "catboost_error_by_cloud_level.csv")
save_table(by_ww, "catboost_error_by_ww_class.csv")
save_table(by_weather_context, "catboost_error_by_weather_context.csv")
save_table(by_env, "catboost_error_by_environment_proxy.csv")
save_table(env_desc, "catboost_environment_proxy_description.csv")

print("\nSynth?se globale :")
display(overall.round(4))
print("\nErreurs par parcours :")
display(by_track.round(4))
print("\nErreurs par M_slot :")
display(by_mslot.round(4))
print("\nPires combinaisons parcours x M_slot :")
display(by_track_mslot.head(12).round(4))
print("\nErreurs par niveau de TMRT observ?e :")
display(by_tmrt.round(4))
print("\nErreurs par m?t?o :")
display(by_glo.round(4))
display(by_rain.round(4))
display(by_cloud.round(4))
display(by_ww.round(4))
print("\nErreurs par proxy environnement AlphaEarth :")
display(env_desc.round(4))

# Figures compactes pour le rapport/la pr?sentation.
fig, ax = plt.subplots(figsize=(7, 4))
plot_track = by_track.sort_values("rmse")
ax.barh(plot_track["track_id"], plot_track["rmse"], color="#3566a8")
ax.set_xlabel("RMSE (?C)")
ax.set_title("Erreur par parcours")
ax.grid(axis="x", alpha=0.25)
save_fig(fig, "catboost_error_by_track.png")

pivot = by_track_mslot.pivot(index="track_id", columns="M_slot", values="rmse")
pivot = pivot.reindex(index=[t for t in TRACK_ORDER if t in pivot.index], columns=[m for m in MSLOT_ORDER if m in pivot.columns])
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.to_numpy(dtype=float), cmap="magma", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.iloc[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{val:.1f}", ha="center", va="center", color="white", fontsize=9)
ax.set_title("RMSE par parcours et M_slot")
fig.colorbar(im, ax=ax, label="RMSE (?C)")
save_fig(fig, "catboost_error_heatmap_track_mslot.png")

fig, ax = plt.subplots(figsize=(9, 4))
plot_tmrt = by_tmrt.sort_values("tmrt_level")
ax.bar(plot_tmrt["tmrt_level"], plot_tmrt["bias_pred_minus_obs"], color="#c44e52", alpha=0.85, label="Biais")
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Biais pred - obs (?C)")
ax.set_title("Biais selon le niveau de TMRT observ?e")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "catboost_bias_by_tmrt_level.png")

fig, ax = plt.subplots(figsize=(8, 4))
plot_glo = by_glo.sort_values("glo_level")
ax.bar(plot_glo["glo_level"], plot_glo["rmse"], color="#dd8a2d")
ax.set_ylabel("RMSE (?C)")
ax.set_title("Erreur selon le rayonnement global")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "catboost_error_by_glo_level.png")

fig, ax = plt.subplots(figsize=(9, 4))
plot_env = env_desc.sort_values("environment_proxy")
ax.bar(plot_env["environment_proxy"], plot_env["rmse"], color="#4c956c")
ax.set_ylabel("RMSE (?C)")
ax.set_title("Erreur selon le proxy environnement AlphaEarth")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "catboost_error_by_environment_proxy.png")

sample = test_errors.sample(n=min(30000, len(test_errors)), random_state=RANDOM_STATE)
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(sample["lon_std"], sample["lat_std"], c=sample["err"], s=4, cmap="coolwarm", vmin=-10, vmax=10, alpha=0.75)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Carte des r?sidus CatBoost (test day_track)")
fig.colorbar(sc, ax=ax, label="pred - obs (?C)")
save_fig(fig, "catboost_spatial_residual_map_day_track.png")



### R?sultat de la piste 12

Les exports d?taill?s sont dans `data/processed/prediction/error_analysis/` et les figures dans `prediction/figures/model_analysis/`.

Principaux constats sur le test `day_track_group` :

- performance globale : RMSE 5,93 ?C, MAE 3,69 ?C, biais moyen `pred - obs` de -1,08 ?C ;
- par parcours, l'?cusson est le plus difficile : RMSE 6,51 ?C et biais -2,31 ?C, donc le mod?le sous-pr?dit fortement ce parcours ;
- par cr?neau, M2 est le plus difficile : RMSE 7,70 ?C, suivi de M3 ? 5,87 ?C ; M4 est beaucoup plus facile ;
- les pires groupes sont surtout `ecusson/M2`, `boulevards/M2`, `antigone/M3` et `ecusson/M3` ;
- par niveau de TMRT, le mod?le sous-pr?dit fortement les plus fortes TMRT : sur le quintile haut, biais -7,77 ?C et RMSE 11,30 ?C ; il sur-pr?dit les tr?s faibles TMRT ;
- par m?t?o, l'erreur augmente fortement quand le rayonnement global est fort ou tr?s fort ; les situations pluvieuses sont rares et plus faciles car la TMRT y varie moins ;
- le proxy AlphaEarth `ouvert/expos?` est le plus difficile, mais certains profils plus ombrag?s ont aussi un biais n?gatif. Ces profils sont des clusters d'embeddings ? interpr?ter visuellement, pas des classes urbaines valid?es.

Conclusion : le mod?le lisse trop les extr?mes. Il pr?dit correctement le centre de la distribution, mais il manque les pics de TMRT, surtout en conditions tr?s ensoleill?es et sur certains passages urbains comme l'?cusson.

